# Stage 02_pcm_fixed_points: pcm_fixed_points
Configuration, inputs and results remain in the corresponding stages directory.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
pcm_fixed_points_experiment_dir = ROOT / 'experiment/research/gap_mechanisms/stages/02_pcm_fixed_points'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

pcm_fixed_points_data_paths = ExperimentPaths(ROOT, pcm_fixed_points_experiment_dir / 'config/paths.json')


# Logical near-degeneracy -> PCM-dependent BP fixed points

This stage implements Parts A–E. Mechanism experiments use a gap-enrichment cohort; final LER uses fixed unbiased held-out test shots. Oracle fields are restricted to offline attribution. Selectors and the final policy are frozen using only the development split.


In [ ]:
from dataclasses import dataclass, replace
from itertools import combinations, product
from math import comb
from pathlib import Path
from time import perf_counter
import json
import sys
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import csc_matrix
from scipy.special import logsumexp
from scipy.stats import spearmanr, wilcoxon
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from decoders.cpp_backend import Tesseract
from decoders.cython.cython_core.osd_core import ordered_statistics_postprocess
from noises_ca import CircuitBuilder, ErrorRate

pcm_fixed_points_config = json.loads((pcm_fixed_points_experiment_dir / 'config/next_gap_mechanism_pcm_fixed_point_experiments.json').read_text(encoding='utf-8'))
if pcm_fixed_points_config['smoke_mode']:
    pcm_fixed_points_config.update(
        primary_levels=[2],
        validation_levels=[3],
        p_values=[0.05],
        validation_p_values=[0.05],
        minimum_shots_per_point=240,
        maximum_shots_per_point=600,
        sampling_batch_size=120,
        target_gap_cases_per_point=4,
        analysis_gap_limit_per_point=4,
        bp_standard_iterations=8,
        continued_iterations=40,
        configured_full_osd_order_by_distance={'5': 2, '9': 2},
        basis_views=4,
        redundant_views=4,
        redundant_rows_per_view=[1, 2],
        restart_count=8,
        basin_case_limit=4,
        logical_margin_control_limit_per_group=2,
        pcm_budgets=[1, 2],
        diversity_mu_values=[0.0, 0.5],
        selector_eta_values=[0.0, 1.0],
        selector_temperature_values=[0.5, 1.0],
        bootstrap_repetitions=30,
        final_pcm_views=2,
        final_case_limit_per_point=8,
        milp_time_limit_seconds=15.0,
        tesseract_timing_shots_per_batch=12,
        random_seed=20260912,
    )

# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# primary_levels: non-empty.
# validation_levels: non-empty.
# primary_levels, validation_levels: values restricted to {2, 3}.
# p_values, validation_p_values: each item is in (0, 1).
# Expected relationship: 0 < minimum_shots_per_point <= maximum_shots_per_point.
# sampling_batch_size: positive.
# target_gap_cases_per_point: positive.
# Expected relationship: continued_iterations > bp_standard_iterations > 0.
# osd_orders: [0, 1, 2].
# configured_full_osd_order_by_distance: keys exactly {'5', '9'}.
# values of configured_full_osd_order_by_distance: each item is at least 2.
# basis_views: positive.
# redundant_views: positive.
# analysis_gap_limit_per_point: positive.
# candidate_max_row_weight: positive.
# redundant_rows_per_view: non-empty; each item is positive.
# restart_count: positive.
# basin_case_limit: positive.
# restart_message_sigma: non-negative.
# fixed_point_tail_window: at least 2.
# fixed_point_belief_tolerance: positive.
# logical_margin_control_limit_per_group: positive.
# logical_margin_quartiles: at least 2.
# ambiguity_top_k: positive.
# selector_confidence_epsilon: positive.
# pcm_budgets: sorted and unique.
# Expected relationship: minimum of pcm_budgets >= 1.
# Expected relationship: final_pcm_views >= maximum of pcm_budgets.
# targeting_strategies: values exactly {'random', 'instability', 'ambiguity', 'conflict',
# 'hybrid'}.
# targeting_constructions: values exactly {'top_m_distinct', 'progressive',
# 'diversity_constrained'}.
# diversity_mu_values: each item is non-negative.
# selector_eta_values: each item is non-negative.
# selector_temperature_values: each item is positive.
# bootstrap_repetitions: positive.
# development_fraction: in (0, 1).
# final_case_limit_per_point: -1 or positive.
# e1_stream_chunk_shots: positive.
# milp_time_limit_seconds: positive.
# tesseract_beam: positive.
# tesseract_pq_limit: positive.
# tesseract_timing_shots_per_batch: positive.
pcm_fixed_points_name = 'next_gap_mechanism_pcm_fixed_point_smoke' if pcm_fixed_points_config['smoke_mode'] else 'next_gap_mechanism_pcm_fixed_point'
pcm_fixed_points_out = pcm_fixed_points_data_paths.run(smoke_mode=pcm_fixed_points_config['smoke_mode'])

from decoders.cython.cython_core.osd_core import OSD_VERSION, osd_candidate_count
from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(pcm_fixed_points_data_paths, pcm_fixed_points_config, {'osd': OSD_VERSION, 'fixed_point': 'message_state_v1'})

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(pcm_fixed_points_data_paths, pcm_fixed_points_config)

pcm_fixed_points_conf, pcm_fixed_points_trace_dir, pcm_fixed_points_stats = (pcm_fixed_points_data_paths.output('config'), pcm_fixed_points_data_paths.output('trace'), pcm_fixed_points_data_paths.output('stats'))

for pcm_fixed_points_directory in [pcm_fixed_points_conf, pcm_fixed_points_trace_dir, pcm_fixed_points_stats]:
    pcm_fixed_points_directory.mkdir(parents=True, exist_ok=True)

pcm_fixed_points_config_path = pcm_fixed_points_data_paths.output('config_json')
if not pcm_fixed_points_config_path.is_file() or not pcm_fixed_points_config['reuse_existing_data']:
    pcm_fixed_points_config_path.write_text(json.dumps({'config': pcm_fixed_points_config}, indent=2), encoding='utf-8')

from experiment.utils import misc

def pcm_fixed_points_log(level, payload):
    return misc.log(level, payload, CONFIG=pcm_fixed_points_config)

def pcm_fixed_points_stage_cached(paths):
    paths = [Path(path) for path in paths]
    present = [path.is_file() for path in paths]
    if pcm_fixed_points_config['reuse_existing_data']:
        assert all(present) or not any(present), f'partial cache: {[str(p) for p, ok in zip(paths, present) if ok]}'

    return pcm_fixed_points_config['reuse_existing_data'] and all(present)

pcm_fixed_points_bit_columns = ['syndrome', 'truth', 'error', 'correction', 'logical_sector', 'standard_bp_correction', 'standard_bp_sector', 'tesseract_sector', 'selected_correction', 'true_sector']

def pcm_fixed_points_read_csv(path):
    columns = pd.read_csv(path, nrows=0).columns
    dtype = {column: str for column in pcm_fixed_points_bit_columns + ['shot_id', 'candidate_id', 'pair_id'] if column in columns}
    return pd.read_csv(path, dtype=dtype, low_memory=False)

pcm_fixed_points_save_csv = misc.save_csv
pcm_fixed_points_bits = misc.bits
pcm_fixed_points_bit_array = misc.bit_array_variant_3

from experiment.utils import gap_mechanisms

def pcm_fixed_points_split_for(shot_id):
    return gap_mechanisms.split_for(shot_id, CONFIG=pcm_fixed_points_config)

sns.set_theme(style='whitegrid')
pcm_fixed_points_log(2, {'output': str(pcm_fixed_points_out)})

assert pcm_fixed_points_config['fixed_point_tail_window'] >= 2
for key in ['fixed_point_message_tolerance', 'fixed_point_state_tolerance']:
    assert np.isfinite(pcm_fixed_points_config[key]) and pcm_fixed_points_config[key] > 0


## Shared BP, OSD, logical-sector and Tesseract primitives

Initialize the shared native-BP trace, OSD and logical-sector operations. All PCM views must describe the same original syndrome constraint and logical decoding problem.


In [ ]:
from experiment.utils import gap_mechanisms

pcm_fixed_points_CodeContext = gap_mechanisms.CodeContext
pcm_fixed_points_BPTrace = gap_mechanisms.BPTrace
pcm_fixed_points_contexts = {}
pcm_fixed_points_gf2_rank = gap_mechanisms.gf2_rank
pcm_fixed_points_graph_adjacency = gap_mechanisms.graph_adjacency

def pcm_fixed_points_context_for(level, p):
    return gap_mechanisms.context_for(
        level,
        p,
        CA_9_2_5=CA_9_2_5,
        CONTEXTS=pcm_fixed_points_contexts,
        CircuitBuilder=CircuitBuilder,
        CodeContext=pcm_fixed_points_CodeContext,
        ErrorRate=ErrorRate,
        TriangularCode=TriangularCode,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
        gf2_rank=pcm_fixed_points_gf2_rank,
        graph_adjacency=pcm_fixed_points_graph_adjacency,
    )

def pcm_fixed_points_transformed_context(context, transform):
    return gap_mechanisms.transformed_context(context, transform, graph_adjacency=pcm_fixed_points_graph_adjacency)

def pcm_fixed_points_trace_bp(
    context,
    syndrome,
    iterations,
    update_rule='min_sum',
    scaling=1.0,
    offset=0.0,
    damping=0.0,
    schedule='parallel',
    rng=None,
    prior_llr=None,
    initial_message_sigma=0.0,
):
    return gap_mechanisms.trace_bp(context, syndrome, iterations, update_rule, scaling, offset, damping, schedule, rng, prior_llr, initial_message_sigma, BPTrace=pcm_fixed_points_BPTrace)

def pcm_fixed_points_logical_label(context, correction):
    return gap_mechanisms.logical_label(context, correction, bits=pcm_fixed_points_bits)

def pcm_fixed_points_outcome(
    context,
    correction,
    syndrome,
    truth,
    error=None,
):
    return gap_mechanisms.outcome(context, correction, syndrome, truth, error, bits=pcm_fixed_points_bits)

pcm_fixed_points_prior_cost = gap_mechanisms.prior_cost

def pcm_fixed_points_full_osd_order(context):
    return gap_mechanisms.full_osd_order(context, CONFIG=pcm_fixed_points_config)

def pcm_fixed_points_osd_decode(
    context,
    syndrome,
    llr,
    order,
):
    return gap_mechanisms.osd_decode(context, syndrome, llr, order, ordered_statistics_postprocess=ordered_statistics_postprocess)

def pcm_fixed_points_standard_bposd(context, syndrome, trace):
    return gap_mechanisms.standard_bposd(context, syndrome, trace, CONFIG=pcm_fixed_points_config, full_osd_order=pcm_fixed_points_full_osd_order, osd_decode=pcm_fixed_points_osd_decode)

pcm_fixed_points_trace_metrics = gap_mechanisms.trace_metrics

def pcm_fixed_points_tesseract_predictions(context, syndromes):
    return gap_mechanisms.tesseract_predictions(context, syndromes, CONFIG=pcm_fixed_points_config, Tesseract=Tesseract)

def pcm_fixed_points_exact_sector_costs(context, syndrome):
    return gap_mechanisms.exact_sector_costs(context, syndrome, CONFIG=pcm_fixed_points_config, bits=pcm_fixed_points_bits)


In [ ]:
def bp_endpoint(trace, budget=None):
    budget = pcm_fixed_points_config['bp_standard_iterations'] if budget is None else int(budget)
    valid = np.flatnonzero(trace.valid[1:budget + 1]) + 1
    return int(valid[0]) if len(valid) else budget

def fixed_point_classification(trace):
    window = pcm_fixed_points_config['fixed_point_tail_window']
    assert window >= 2 and len(trace.hard) >= window + 1
    tail_hard = trace.hard[-window:]
    belief_step = float(np.max(np.linalg.norm(np.diff(trace.llr[-window:], axis=0), axis=1)))
    message_step = max(float(np.max(np.abs(np.diff(values[-window:], axis=0))))
                       for values in (trace.c2v, trace.v2c))
    belief_tolerance = pcm_fixed_points_config['fixed_point_belief_tolerance']
    message_tolerance = pcm_fixed_points_config['fixed_point_message_tolerance']
    if np.all(tail_hard == tail_hard[-1]) and belief_step <= belief_tolerance and message_step <= message_tolerance:
        return ('fixed_point', belief_step)
    if len(trace.hard) >= 4:
        belief_repeat = float(np.max(np.linalg.norm(trace.llr[-2:] - trace.llr[-4:-2], axis=1)))
        message_repeat = max(float(np.max(np.abs(values[-2:] - values[-4:-2])))
                             for values in (trace.c2v, trace.v2c))
        if np.array_equal(trace.hard[-4:-2], trace.hard[-2:]) and belief_repeat <= belief_tolerance and message_repeat <= message_tolerance:
            return ('period_2', belief_repeat)
    return ('unfinished', belief_step)


def fixed_point_cluster(trace, clusters):
    # Complete-link membership avoids joining a chain of distant endpoints.
    state = np.concatenate([trace.llr[-1], trace.c2v[-1], trace.v2c[-1]])
    hard = trace.hard[-1].copy()
    tolerance = pcm_fixed_points_config['fixed_point_state_tolerance']
    for cluster_id, members in enumerate(clusters):
        if all(np.array_equal(hard, old_hard) and np.max(np.abs(state - old_state)) <= tolerance
               for old_state, old_hard in members):
            members.append((state, hard))
            return cluster_id
    clusters.append([(state, hard)])
    return len(clusters) - 1

def entropy_from_counts(counts):
    probabilities = np.asarray(counts, dtype=float)
    probabilities = probabilities[probabilities > 0]
    probabilities /= probabilities.sum()

    return float(-(probabilities * np.log(probabilities)).sum())

def variable_diagnostics(context, trace, endpoint):
    start = max(1, endpoint - max(2, endpoint // 5) + 1)
    ambiguity = 1 / (np.abs(trace.llr[endpoint]) + pcm_fixed_points_config['selector_confidence_epsilon'])
    instability = np.zeros(context.H.shape[1])
    conflict = np.zeros(context.H.shape[1])

    for variable, ids in enumerate(context.variable_edges):
        incoming = trace.c2v[start:endpoint + 1, ids]
        instability[variable] = np.mean(np.abs(np.diff(trace.c2v[start - 1:endpoint + 1, ids], axis=0)))
        denom = np.abs(incoming).sum(axis=1)
        conflict[variable] = np.mean(1 - np.abs(incoming.sum(axis=1)) / (denom + 1e-12))

    return (ambiguity, instability, conflict)

def row_candidates(context):
    rows, seen = ([], set())
    m = context.H.shape[0]
    for left, right in combinations(range(m), 2):
        coefficient = np.zeros(m, dtype=np.uint8)
        coefficient[[left, right]] = 1
        row = coefficient @ context.H % 2
        key = row.tobytes()

        if 0 < row.sum() <= pcm_fixed_points_config['candidate_max_row_weight'] and key not in seen:
            seen.add(key)
            rows.append((coefficient, row))

    assert len(rows) >= max(pcm_fixed_points_config['pcm_budgets']), 'insufficient distinct redundant rows'
    return rows

def random_basis_transforms(context, count, rng):
    m = context.H.shape[0]
    results, seen = ([], set())
    attempts = 0

    while len(results) < count:
        transform = np.eye(m, dtype=np.uint8)
        for _ in range(int(rng.integers(1, max(2, m)))):
            left, right = rng.choice(m, 2, replace=False)
            transform[left] ^= transform[right]

        key = transform.tobytes()
        attempts += 1
        if key not in seen:
            assert pcm_fixed_points_gf2_rank(transform) == m
            seen.add(key)
            results.append(transform)
        assert attempts < 100 * count

    return results

def random_redundant_transforms(context, count, rng):
    m = context.H.shape[0]
    candidates = row_candidates(context)
    results, seen = ([], set())
    attempts = 0

    while len(results) < count:
        amount = min(int(rng.choice(pcm_fixed_points_config['redundant_rows_per_view'])), len(candidates))
        selected = rng.choice(len(candidates), amount, replace=False)
        coefficients = np.asarray([candidates[int(index)][0] for index in np.atleast_1d(selected)])
        transform = np.vstack([np.eye(m, dtype=np.uint8), coefficients])
        key = transform.tobytes()
        attempts += 1

        if key not in seen:
            seen.add(key)
            results.append(transform)
        assert attempts < 100 * count

    return results

def decode_view(
    context,
    syndrome,
    truth,
    transform,
    candidate_id,
    view_kind,
    source,
    selected_support='',
):
    view_context = pcm_fixed_points_transformed_context(context, transform)
    view_syndrome = transform @ syndrome % 2
    assert np.array_equal(view_context.H, transform @ context.H % 2)
    trace = pcm_fixed_points_trace_bp(view_context, view_syndrome, pcm_fixed_points_config['bp_standard_iterations'])
    endpoint = bp_endpoint(trace)
    correction = trace.hard[endpoint]
    valid, correct, _, sector = pcm_fixed_points_outcome(context, correction, syndrome, truth)
    metrics = pcm_fixed_points_trace_metrics(view_context, trace, pcm_fixed_points_config['bp_standard_iterations'])
    convergence_class, terminal_belief_step = fixed_point_classification(trace)
    ambiguity = float(np.mean(1 / (np.abs(trace.llr[endpoint]) + pcm_fixed_points_config['selector_confidence_epsilon'])))

    return {
        'candidate_id': candidate_id,
        'view_kind': view_kind,
        'pcm_source': source,
        'transform_json': json.dumps(transform.tolist()),
        'selected_row_support': selected_support,
        'syndrome_consistent': valid,
        'logical_correct': correct,
        'logical_sector': sector,
        'correction': pcm_fixed_points_bits(correction),
        'llr_json': json.dumps(trace.llr[endpoint].tolist()),
        'first_valid_iteration': endpoint if valid else -1,
        'syndrome_reached': valid,
        'converged': int(convergence_class == 'fixed_point'),
        'convergence_class': convergence_class,
        'terminal_belief_step': terminal_belief_step,
        'iterations': endpoint,
        'dynamics_horizon': pcm_fixed_points_config['bp_standard_iterations'],
        'prior_cost': pcm_fixed_points_prior_cost(context, correction),
        'bp_confidence': metrics['posterior_abs_mean'],
        'view_ambiguity': ambiguity,
        'runtime': float(trace.runtime[endpoint]),
        **metrics,
        **graph_metrics_extended(view_context.H),
    }

def tanner_graph_local(H):
    graph = nx.Graph()
    graph.add_nodes_from([f'c{i}' for i in range(H.shape[0])], bipartite=0)
    graph.add_nodes_from([f'v{i}' for i in range(H.shape[1])], bipartite=1)
    checks, variables = np.nonzero(H)
    graph.add_edges_from(((f'c{c}', f'v{v}') for c, v in zip(checks, variables)))

    return graph

def cycle_counts_extended(H):
    shared = H @ H.T
    four = int(sum((comb(int(value), 2) for value in shared[np.triu_indices(H.shape[0], 1)] if value >= 2)))
    supports = [set(np.flatnonzero(H[row])) for row in range(H.shape[0])]
    six = 0

    for first, second, third in combinations(range(H.shape[0]), 3):
        for v12 in supports[first] & supports[second]:
            for v23 in supports[second] & supports[third]:
                if v23 != v12:
                    six += sum((v31 not in {v12, v23} for v31 in supports[third] & supports[first]))

    graph = tanner_graph_local(H)
    girth = nx.girth(graph)

    return (four, int(six), np.nan if not np.isfinite(girth) else int(girth), int(graph.number_of_edges() - graph.number_of_nodes() + nx.number_connected_components(graph)))

def graph_metrics_extended(H):
    four, six, girth, cycle_rank = cycle_counts_extended(H)
    return {
        'row_count': H.shape[0],
        'edge_count': int(H.sum()),
        'four_cycle_count': four,
        'six_cycle_count': six,
        'girth': girth,
        'cycle_rank': cycle_rank,
        'mean_variable_degree': float(H.sum(axis=0).mean()),
        'mean_check_degree': float(H.sum(axis=1).mean()),
    }

def local_graph_metrics_extended(H, variables, hops):
    variables = set(map(int, variables))
    checks = set()
    for _ in range(hops):
        checks |= set(np.flatnonzero(np.any(H[:, list(variables)], axis=1))) if variables else set()
        variables |= set(np.flatnonzero(np.any(H[list(checks)], axis=0))) if checks else set()

    if not variables or not checks:
        return {
            'local_variables': len(variables),
            'local_checks': len(checks),
            'local_four_cycles': 0,
            'local_six_cycles': 0,
            'local_cycle_rank': 0,
            'local_girth': np.nan,
        }

    sub = H[np.ix_(sorted(checks), sorted(variables))]
    four, six, girth, rank = cycle_counts_extended(sub)

    return {
        'local_variables': len(variables),
        'local_checks': len(checks),
        'local_four_cycles': four,
        'local_six_cycles': six,
        'local_cycle_rank': rank,
        'local_girth': girth,
    }


## Fixed unbiased population and gap-enrichment cohort

Construct the unbiased physical-shot population and the separately enriched gap cohort. The mechanism blocks and final LER block use different populations and must keep their denominators separate.


In [ ]:
pcm_fixed_points_population_path = pcm_fixed_points_data_paths.output('population_csv')
if pcm_fixed_points_stage_cached([pcm_fixed_points_population_path]):
    pcm_fixed_points_population = pcm_fixed_points_read_csv(pcm_fixed_points_population_path)
else:
    pcm_fixed_points_rows = []
    specs = [(level, p, 'primary') for level in pcm_fixed_points_config['primary_levels'] for p in pcm_fixed_points_config['p_values']]
    specs += [(level, p, 'validation') for level in pcm_fixed_points_config['validation_levels'] for p in pcm_fixed_points_config['validation_p_values']]

    for pcm_fixed_points_point_index, (pcm_fixed_points_level, pcm_fixed_points_p, pcm_fixed_points_point_role) in enumerate(specs):
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_level, pcm_fixed_points_p)
        pcm_fixed_points_point_rows = []
        pcm_fixed_points_seen = 0

        while pcm_fixed_points_seen < pcm_fixed_points_config['maximum_shots_per_point']:
            pcm_fixed_points_amount = min(pcm_fixed_points_config['sampling_batch_size'], pcm_fixed_points_config['maximum_shots_per_point'] - pcm_fixed_points_seen)
            pcm_fixed_points_sampler = pcm_fixed_points_context.dem.compile_sampler(seed=pcm_fixed_points_config['random_seed'] + pcm_fixed_points_point_index * 100003 + pcm_fixed_points_seen)
            pcm_fixed_points_syndromes, pcm_fixed_points_truths, pcm_fixed_points_errors = pcm_fixed_points_sampler.sample(shots=pcm_fixed_points_amount, return_errors=True)
            pcm_fixed_points_tess, pcm_fixed_points_tess_runtime, tess_compile = pcm_fixed_points_tesseract_predictions(pcm_fixed_points_context, pcm_fixed_points_syndromes)
            pcm_fixed_points_iterator = zip(pcm_fixed_points_syndromes, pcm_fixed_points_truths, pcm_fixed_points_errors, pcm_fixed_points_tess, pcm_fixed_points_tess_runtime)

            for pcm_fixed_points_local, pcm_fixed_points_values in enumerate(tqdm(pcm_fixed_points_iterator, total=pcm_fixed_points_amount, desc=f'population d={pcm_fixed_points_context.distance} p={pcm_fixed_points_p:g}', disable=pcm_fixed_points_config['print_level'] == 1)):
                pcm_fixed_points_syndrome, pcm_fixed_points_truth, pcm_fixed_points_error, pcm_fixed_points_prediction = [np.asarray(value, dtype=np.uint8) for value in pcm_fixed_points_values[:4]]
                pcm_fixed_points_trace = pcm_fixed_points_trace_bp(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_config['bp_standard_iterations'])
                bp_end = bp_endpoint(pcm_fixed_points_trace)
                pcm_fixed_points_standard, standard_end, osd_used, pcm_fixed_points_standard_runtime = pcm_fixed_points_standard_bposd(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_trace)
                pcm_fixed_points_bp_valid, bp_correct, pcm_fixed_points_, bp_sector = pcm_fixed_points_outcome(pcm_fixed_points_context, pcm_fixed_points_trace.hard[bp_end], pcm_fixed_points_syndrome, pcm_fixed_points_truth)
                standard_valid, standard_correct, pcm_fixed_points_exact, standard_sector = pcm_fixed_points_outcome(pcm_fixed_points_context, pcm_fixed_points_standard, pcm_fixed_points_syndrome, pcm_fixed_points_truth, pcm_fixed_points_error)
                pcm_fixed_points_metrics = pcm_fixed_points_trace_metrics(pcm_fixed_points_context, pcm_fixed_points_trace, bp_end)
                pcm_fixed_points_shot_id = f'L{pcm_fixed_points_level}-p{pcm_fixed_points_point_index}-b{pcm_fixed_points_seen}-i{pcm_fixed_points_local}'

                pcm_fixed_points_point_rows.append({
                    'shot_id': pcm_fixed_points_shot_id,
                    'level': pcm_fixed_points_level,
                    'distance': pcm_fixed_points_context.distance,
                    'p': pcm_fixed_points_p,
                    'point_role': pcm_fixed_points_point_role,
                    'population_role': 'unbiased' if pcm_fixed_points_seen + pcm_fixed_points_local < pcm_fixed_points_config['minimum_shots_per_point'] else 'gap_enrichment',
                    'split': pcm_fixed_points_split_for(pcm_fixed_points_shot_id),
                    'syndrome': pcm_fixed_points_bits(pcm_fixed_points_syndrome),
                    'truth': pcm_fixed_points_bits(pcm_fixed_points_truth),
                    'error': pcm_fixed_points_bits(pcm_fixed_points_error),
                    'error_weight': int(pcm_fixed_points_error.sum()),
                    'standard_bp_valid': pcm_fixed_points_bp_valid,
                    'standard_bp_correct': bp_correct,
                    'standard_bp_sector': bp_sector,
                    'standard_bp_endpoint': bp_end,
                    'standard_bp_correction': pcm_fixed_points_bits(pcm_fixed_points_trace.hard[bp_end]),
                    'standard_bp_llr_json': json.dumps(pcm_fixed_points_trace.llr[bp_end].tolist()),
                    'standard_bp_runtime': float(pcm_fixed_points_trace.runtime[bp_end]),
                    'standard_valid': standard_valid,
                    'standard_correct': standard_correct,
                    'standard_sector': standard_sector,
                    'standard_exact': pcm_fixed_points_exact,
                    'standard_endpoint': standard_end,
                    'standard_osd_used': osd_used,
                    'standard_runtime': pcm_fixed_points_standard_runtime,
                    'tesseract_correct': int(np.array_equal(pcm_fixed_points_prediction, pcm_fixed_points_truth)),
                    'tesseract_sector': pcm_fixed_points_bits(pcm_fixed_points_prediction),
                    'tesseract_runtime': float(pcm_fixed_points_values[4]),
                    'tesseract_compile_amortized': tess_compile,
                    **pcm_fixed_points_metrics,
                })

            pcm_fixed_points_seen += pcm_fixed_points_amount
            pcm_fixed_points_frame = pd.DataFrame(pcm_fixed_points_point_rows)
            gaps = int(((pcm_fixed_points_frame.standard_correct == 0) & (pcm_fixed_points_frame.tesseract_correct == 1)).sum())
            pcm_fixed_points_log(2, {
                'stage': 'population',
                'distance': pcm_fixed_points_context.distance,
                'p': pcm_fixed_points_p,
                'shots': pcm_fixed_points_seen,
                'gap': gaps,
            })
            pcm_fixed_points_enough_gap = gaps >= pcm_fixed_points_config['target_gap_cases_per_point'] if pcm_fixed_points_point_role == 'primary' else True

            if pcm_fixed_points_seen >= pcm_fixed_points_config['minimum_shots_per_point'] and pcm_fixed_points_enough_gap:
                break

        pcm_fixed_points_rows.extend(pcm_fixed_points_point_rows)

    pcm_fixed_points_population = pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_rows), pcm_fixed_points_population_path)

pcm_fixed_points_population['gap'] = ((pcm_fixed_points_population.standard_correct == 0) & (pcm_fixed_points_population.tesseract_correct == 1)).astype(int)
unbiased = pcm_fixed_points_population[pcm_fixed_points_population.population_role == 'unbiased'].copy()
assert unbiased.groupby(['distance', 'p']).size().eq(pcm_fixed_points_config['minimum_shots_per_point']).all()
pcm_fixed_points_primary_distances = {pcm_fixed_points_context_for(level, pcm_fixed_points_config['p_values'][0]).distance for level in pcm_fixed_points_config['primary_levels']}
pcm_fixed_points_gap_cases = pcm_fixed_points_population[(pcm_fixed_points_population.gap == 1) & pcm_fixed_points_population.distance.isin(pcm_fixed_points_primary_distances)].groupby(['distance', 'p'], sort=False).head(pcm_fixed_points_config['analysis_gap_limit_per_point']).reset_index(drop=True)
assert len(pcm_fixed_points_gap_cases), 'No gap cases collected'
assert set(unbiased[unbiased.split == 'development'].shot_id).isdisjoint(set(unbiased[unbiased.split == 'test'].shot_id))
pcm_fixed_points_save_csv(
    unbiased.groupby(['distance', 'p'], as_index=False).agg(
        shots=('shot_id', 'size'),
        standard_LER=('standard_correct', lambda x: 1 - x.mean()),
        tesseract_LER=('tesseract_correct', lambda x: 1 - x.mean()),
        gap_rate=('gap', 'mean'),
    ),
    pcm_fixed_points_data_paths.output('population_summary_csv'),
)


## Exact logical-sector margins for gap and control cohorts

Compute exact sector margins for the selected gap and control cases. Smaller top-sector separation indicates posterior competition under this prior model; it does not directly prescribe a runtime selector.


In [ ]:
MARGIN_SECTOR_PATH = pcm_fixed_points_trace_dir / 'logical_sector_costs.csv'
MARGIN_PATH = pcm_fixed_points_trace_dir / 'logical_margins.csv'
if pcm_fixed_points_stage_cached([MARGIN_SECTOR_PATH, MARGIN_PATH]):
    margin_sectors = pcm_fixed_points_read_csv(MARGIN_SECTOR_PATH)
    pcm_fixed_points_margins = pcm_fixed_points_read_csv(MARGIN_PATH)
else:
    controls = unbiased.copy()
    controls['cohort'] = np.select([(controls.standard_correct == 1) & (controls.tesseract_correct == 1), (controls.standard_correct == 0) & (controls.tesseract_correct == 0)], ['both_correct', 'both_wrong'], default='other')
    controls = controls[controls.cohort != 'other'].groupby(['distance', 'p', 'cohort'], sort=False).head(pcm_fixed_points_config['logical_margin_control_limit_per_group'])
    gaps = pcm_fixed_points_gap_cases.copy()
    gaps['cohort'] = 'gap'
    margin_population = pd.concat([controls, gaps], ignore_index=True).drop_duplicates('shot_id')
    sector_rows, margin_rows = ([], [])

    for pcm_fixed_points_case in tqdm(margin_population.itertuples(index=False), total=len(margin_population), desc='logical margins', disable=pcm_fixed_points_config['print_level'] == 1):
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_case.level, pcm_fixed_points_case.p)
        pcm_fixed_points_syndrome = pcm_fixed_points_bit_array(pcm_fixed_points_case.syndrome, pcm_fixed_points_context.H.shape[0])
        pcm_fixed_points_truth = pcm_fixed_points_bit_array(pcm_fixed_points_case.truth, pcm_fixed_points_context.L.shape[0])
        pcm_fixed_points_costs = pcm_fixed_points_exact_sector_costs(pcm_fixed_points_context, pcm_fixed_points_syndrome)
        assert all((row['exact'] for row in pcm_fixed_points_costs))
        true_sector = pcm_fixed_points_bits(pcm_fixed_points_truth)
        pcm_fixed_points_true_cost = next((row['cost'] for row in pcm_fixed_points_costs if row['sector'] == true_sector))
        pcm_fixed_points_competitor = min((row['cost'] for row in pcm_fixed_points_costs if row['sector'] != true_sector))
        margin = pcm_fixed_points_competitor - pcm_fixed_points_true_cost
        pcm_fixed_points_scale = float(np.median(np.abs(pcm_fixed_points_context.prior_llr)))
        margin_rows.append({
            'shot_id': pcm_fixed_points_case.shot_id,
            'distance': pcm_fixed_points_case.distance,
            'p': pcm_fixed_points_case.p,
            'split': pcm_fixed_points_case.split,
            'cohort': pcm_fixed_points_case.cohort,
            'logical_margin': margin,
            'absolute_logical_margin': abs(margin),
            'normalized_logical_margin': abs(margin) / pcm_fixed_points_scale,
        })

        for pcm_fixed_points_row in pcm_fixed_points_costs:
            sector_rows.append({
                'shot_id': pcm_fixed_points_case.shot_id,
                'true_sector': true_sector,
                **pcm_fixed_points_row,
            })

    margin_sectors = pcm_fixed_points_save_csv(pd.DataFrame(sector_rows), MARGIN_SECTOR_PATH)
    pcm_fixed_points_margins = pcm_fixed_points_save_csv(pd.DataFrame(margin_rows), MARGIN_PATH)

assert margin_sectors.groupby('shot_id').exact.min().eq(1).all()


## Part A — A1 basis/redundant split and A2 paired analysis

Separate invertible basis replacement from adding redundant rows, then compare successful and failing views of the same shot. Both preserve the constraint set, but graph size, messages and computation can differ.


In [ ]:
A1_PATH = pcm_fixed_points_trace_dir / 'A1_pcm_views.csv'
if pcm_fixed_points_stage_cached([A1_PATH]):
    a1_views = pcm_fixed_points_read_csv(A1_PATH)
else:
    pcm_fixed_points_rows = []
    for pcm_fixed_points_case_index, pcm_fixed_points_case in enumerate(tqdm(pcm_fixed_points_gap_cases.itertuples(index=False), total=len(pcm_fixed_points_gap_cases), desc='A1 PCM split', disable=pcm_fixed_points_config['print_level'] == 1)):
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_case.level, pcm_fixed_points_case.p)
        pcm_fixed_points_syndrome = pcm_fixed_points_bit_array(pcm_fixed_points_case.syndrome, pcm_fixed_points_context.H.shape[0])
        pcm_fixed_points_truth = pcm_fixed_points_bit_array(pcm_fixed_points_case.truth, pcm_fixed_points_context.L.shape[0])
        pcm_fixed_points_rng = np.random.default_rng(pcm_fixed_points_config['random_seed'] + pcm_fixed_points_case_index)
        families = [('basis_replacement', random_basis_transforms(pcm_fixed_points_context, pcm_fixed_points_config['basis_views'], pcm_fixed_points_rng)), ('redundant_addition', random_redundant_transforms(pcm_fixed_points_context, pcm_fixed_points_config['redundant_views'], pcm_fixed_points_rng))]

        for pcm_fixed_points_kind, pcm_fixed_points_transforms in families:
            assert len({transform.tobytes() for transform in pcm_fixed_points_transforms}) == len(pcm_fixed_points_transforms)
            for view_id, pcm_fixed_points_transform in enumerate(pcm_fixed_points_transforms, 1):
                pcm_fixed_points_record = decode_view(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_truth, pcm_fixed_points_transform, f'{pcm_fixed_points_case.shot_id}:{pcm_fixed_points_kind}:{view_id}', pcm_fixed_points_kind, 'A1')
                pcm_fixed_points_record.update(
                    shot_id=pcm_fixed_points_case.shot_id,
                    distance=pcm_fixed_points_case.distance,
                    p=pcm_fixed_points_case.p,
                    split=pcm_fixed_points_case.split,
                    view_id=view_id,
                )
                pcm_fixed_points_rows.append(pcm_fixed_points_record)

    a1_views = pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_rows), A1_PATH)

assert a1_views.groupby(['shot_id', 'view_kind']).candidate_id.nunique().eq(a1_views.groupby(['shot_id', 'view_kind']).size()).all()
a1_case = a1_views.groupby(['shot_id', 'view_kind'], as_index=False).agg(correct_fraction=('logical_correct', 'mean'), oracle_any_correct=('logical_correct', 'max'), sector_count=('logical_sector', 'nunique'))
a1_summary = a1_case.merge(pcm_fixed_points_gap_cases[['shot_id', 'distance', 'p']], on='shot_id').groupby(['distance', 'p', 'view_kind'], as_index=False).agg(cases=('shot_id', 'size'), oracle_generation_rate=('oracle_any_correct', 'mean'), mean_correct_fraction=('correct_fraction', 'mean'))
pcm_fixed_points_save_csv(a1_summary, pcm_fixed_points_data_paths.output('A1_basis_vs_redundant_csv'))
a1_summary


In [ ]:
A2_PAIR_PATH = pcm_fixed_points_trace_dir / 'A2_paired_views.csv'
A2_LOCAL_PATH = pcm_fixed_points_trace_dir / 'A2_paired_local_graph.csv'
if pcm_fixed_points_stage_cached([A2_PAIR_PATH, A2_LOCAL_PATH]):
    a2_pairs = pcm_fixed_points_read_csv(A2_PAIR_PATH)
    a2_local = pcm_fixed_points_read_csv(A2_LOCAL_PATH)
else:
    pcm_fixed_points_pair_rows, local_rows = ([], [])
    for (pcm_fixed_points_shot_id, pcm_fixed_points_kind), pcm_fixed_points_group in a1_views.groupby(['shot_id', 'view_kind'], sort=False):
        pcm_fixed_points_success = pcm_fixed_points_group[pcm_fixed_points_group.logical_correct == 1].sort_values('view_id')
        failure = pcm_fixed_points_group[pcm_fixed_points_group.logical_correct == 0].sort_values('view_id')
        if pcm_fixed_points_success.empty or failure.empty:
            continue
        pcm_fixed_points_successful, pcm_fixed_points_failing = (pcm_fixed_points_success.iloc[0], failure.iloc[0])
        pcm_fixed_points_case = pcm_fixed_points_gap_cases.set_index('shot_id').loc[pcm_fixed_points_shot_id]
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_case.level, pcm_fixed_points_case.p)
        pcm_fixed_points_error = pcm_fixed_points_bit_array(pcm_fixed_points_case.error, pcm_fixed_points_context.H.shape[1])
        bp_hard = pcm_fixed_points_bit_array(pcm_fixed_points_case.standard_bp_correction, pcm_fixed_points_context.H.shape[1])
        repair = pcm_fixed_points_bit_array(pcm_fixed_points_successful.correction, pcm_fixed_points_context.H.shape[1])

        bp_llr = np.asarray(json.loads(pcm_fixed_points_case.standard_bp_llr_json))
        pcm_fixed_points_pair_id = f'{pcm_fixed_points_shot_id}:{pcm_fixed_points_kind}'
        pcm_fixed_points_metrics = ['message_instability', 'message_instability_max', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'four_cycle_count', 'six_cycle_count', 'cycle_rank']
        pcm_fixed_points_pair_rows.append({
            'pair_id': pcm_fixed_points_pair_id,
            'shot_id': pcm_fixed_points_shot_id,
            'view_kind': pcm_fixed_points_kind,
            **{f'delta_{metric}': pcm_fixed_points_successful[metric] - pcm_fixed_points_failing[metric] for metric in pcm_fixed_points_metrics},
        })
        pcm_fixed_points_regions = {
            'oracle_wrong_BP_support': np.flatnonzero(bp_hard ^ pcm_fixed_points_error),
            'least_reliable': np.argsort(np.abs(bp_llr))[:pcm_fixed_points_config['ambiguity_top_k']],
            'oracle_BP_to_repair_difference': np.flatnonzero(bp_hard ^ repair),
        }

        for region, pcm_fixed_points_variables in pcm_fixed_points_regions.items():
            for pcm_fixed_points_hops in [1, 2]:
                fail_H = np.asarray(json.loads(pcm_fixed_points_failing.transform_json), dtype=np.uint8) @ pcm_fixed_points_context.H % 2
                success_H = np.asarray(json.loads(pcm_fixed_points_successful.transform_json), dtype=np.uint8) @ pcm_fixed_points_context.H % 2

                fail_metrics = local_graph_metrics_extended(fail_H, pcm_fixed_points_variables, pcm_fixed_points_hops)
                success_metrics = local_graph_metrics_extended(success_H, pcm_fixed_points_variables, pcm_fixed_points_hops)
                local_rows.append({
                    'pair_id': pcm_fixed_points_pair_id,
                    'shot_id': pcm_fixed_points_shot_id,
                    'view_kind': pcm_fixed_points_kind,
                    'region': region,
                    'hops': pcm_fixed_points_hops,
                    **{f'delta_{key}': success_metrics[key] - fail_metrics[key] for key in success_metrics},
                })

    pcm_fixed_points_pair_columns = ['pair_id', 'shot_id', 'view_kind'] + [f'delta_{metric}' for metric in ['message_instability', 'message_instability_max', 'belief_instability', 'sign_flip', 'extrinsic_conflict', 'four_cycle_count', 'six_cycle_count', 'cycle_rank']]
    local_columns = ['pair_id', 'shot_id', 'view_kind', 'region', 'hops'] + [f'delta_{key}' for key in ['local_variables', 'local_checks', 'local_four_cycles', 'local_six_cycles', 'local_cycle_rank', 'local_girth']]
    a2_pairs = pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_pair_rows, columns=pcm_fixed_points_pair_columns), A2_PAIR_PATH)
    a2_local = pcm_fixed_points_save_csv(pd.DataFrame(local_rows, columns=local_columns), A2_LOCAL_PATH)

def paired_aggregate(frame, group_columns, output):
    rows = []
    value_columns = [column for column in frame if column.startswith('delta_')]
    for keys, group in frame.groupby(group_columns, dropna=False) if len(frame) else []:
        keys = keys if isinstance(keys, tuple) else (keys,)
        for column in value_columns:
            values = group.groupby('shot_id')[column].mean().dropna().to_numpy()
            if not len(values):
                continue
            statistic, pvalue = (np.nan, np.nan) if np.allclose(values, 0) else wilcoxon(values)
            rng = np.random.default_rng(pcm_fixed_points_config['random_seed'] + len(rows))
            indices = rng.integers(0, len(values), size=(pcm_fixed_points_config['bootstrap_repetitions'], len(values)))
            boot = np.median(values[indices], axis=1)
            rows.append({
                **dict(zip(group_columns, keys)),
                'metric': column,
                'shot_clusters': len(values),
                'paired_median': np.median(values),
                'wilcoxon_statistic': statistic,
                'wilcoxon_pvalue': pvalue,
                'bootstrap_ci_low': np.quantile(boot, 0.025),
                'bootstrap_ci_high': np.quantile(boot, 0.975),
            })

    summary_columns = group_columns + ['metric', 'shot_clusters', 'paired_median', 'wilcoxon_statistic', 'wilcoxon_pvalue', 'bootstrap_ci_low', 'bootstrap_ci_high']
    return pcm_fixed_points_save_csv(pd.DataFrame(rows, columns=summary_columns), output)

a2_global_summary = paired_aggregate(a2_pairs, ['view_kind'], pcm_fixed_points_data_paths.output('A2_paired_global_csv'))
a2_local_summary = paired_aggregate(a2_local, ['view_kind', 'region', 'hops'], pcm_fixed_points_data_paths.output('A2_paired_local_csv'))
a2_local_summary


## Part B — margin, multi-basin, PCM sensitivity and oscillation

Test the links among logical margin, restart outcomes, PCM sensitivity and oscillation individually. Correlation along this proposed chain is not sufficient to prove a causal mechanism.


In [ ]:
BASIN_PATH = pcm_fixed_points_trace_dir / 'B2_B3_restart_trials.csv'
if pcm_fixed_points_stage_cached([BASIN_PATH]):
    pcm_fixed_points_restart_trials = pcm_fixed_points_read_csv(BASIN_PATH)
else:
    pcm_fixed_points_rows = []
    selected_for_three = set()
    for pcm_fixed_points_shot_id, pcm_fixed_points_group in a1_views.groupby('shot_id'):
        if pcm_fixed_points_group.logical_correct.any() and (~pcm_fixed_points_group.logical_correct.astype(bool)).any() and (len(selected_for_three) < pcm_fixed_points_config['basin_case_limit']):
            selected_for_three.add(pcm_fixed_points_shot_id)

    for pcm_fixed_points_case_index, pcm_fixed_points_case in enumerate(tqdm(pcm_fixed_points_gap_cases.itertuples(index=False), total=len(pcm_fixed_points_gap_cases), desc='B2-B3 restarts', disable=pcm_fixed_points_config['print_level'] == 1)):
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_case.level, pcm_fixed_points_case.p)
        pcm_fixed_points_syndrome = pcm_fixed_points_bit_array(pcm_fixed_points_case.syndrome, pcm_fixed_points_context.H.shape[0])
        pcm_fixed_points_truth = pcm_fixed_points_bit_array(pcm_fixed_points_case.truth, pcm_fixed_points_context.L.shape[0])
        pcm_fixed_points_transforms = [('standard', np.eye(pcm_fixed_points_context.H.shape[0], dtype=np.uint8))]

        if pcm_fixed_points_case.shot_id in selected_for_three:
            pcm_fixed_points_group = a1_views[a1_views.shot_id == pcm_fixed_points_case.shot_id]
            for pcm_fixed_points_role, pcm_fixed_points_chosen in [('successful', pcm_fixed_points_group[pcm_fixed_points_group.logical_correct == 1].sort_values('candidate_id').iloc[0]), ('failing', pcm_fixed_points_group[pcm_fixed_points_group.logical_correct == 0].sort_values('candidate_id').iloc[0])]:
                pcm_fixed_points_transforms.append((pcm_fixed_points_role, np.asarray(json.loads(pcm_fixed_points_chosen.transform_json), dtype=np.uint8)))

        for pcm_role, pcm_fixed_points_transform in pcm_fixed_points_transforms:
            pcm_fixed_points_view_context = pcm_fixed_points_transformed_context(pcm_fixed_points_context, pcm_fixed_points_transform)
            pcm_fixed_points_view_syndrome = pcm_fixed_points_transform @ pcm_fixed_points_syndrome % 2
            fixed_point_clusters = []
            for pcm_fixed_points_restart in range(pcm_fixed_points_config['restart_count']):
                pcm_fixed_points_rng = np.random.default_rng(pcm_fixed_points_config['random_seed'] + pcm_fixed_points_case_index * 100003 + pcm_fixed_points_restart)
                pcm_fixed_points_trace = pcm_fixed_points_trace_bp(
                    pcm_fixed_points_view_context,
                    pcm_fixed_points_view_syndrome,
                    pcm_fixed_points_config['continued_iterations'],
                    rng=pcm_fixed_points_rng,
                    prior_llr=pcm_fixed_points_context.prior_llr,
                    initial_message_sigma=pcm_fixed_points_config['restart_message_sigma'],
                    schedule='parallel',
                )
                convergence_class, terminal_belief_step = fixed_point_classification(pcm_fixed_points_trace)
                pcm_fixed_points_endpoint = pcm_fixed_points_config['continued_iterations']
                pcm_fixed_points_correction = pcm_fixed_points_trace.hard[pcm_fixed_points_endpoint]
                pcm_fixed_points_valid, pcm_fixed_points_correct, pcm_fixed_points_, pcm_fixed_points_sector = pcm_fixed_points_outcome(pcm_fixed_points_context, pcm_fixed_points_correction, pcm_fixed_points_syndrome, pcm_fixed_points_truth)
                eligible = convergence_class == 'fixed_point' and bool(pcm_fixed_points_valid)
                cluster_id = fixed_point_cluster(pcm_fixed_points_trace, fixed_point_clusters) if eligible else -1
                terminal_message_step = max(float(np.max(np.abs(np.diff(values[-pcm_fixed_points_config['fixed_point_tail_window']:], axis=0)))) for values in (pcm_fixed_points_trace.c2v, pcm_fixed_points_trace.v2c))
                pcm_fixed_points_rows.append({
                    'shot_id': pcm_fixed_points_case.shot_id,
                    'split': pcm_fixed_points_case.split,
                    'pcm_role': pcm_role,
                    'restart': pcm_fixed_points_restart,
                    'restart_kind': 'gaussian_message_initialization',
                    'convergence_class': convergence_class,
                    'terminal_belief_step': terminal_belief_step,
                    'syndrome_consistent': pcm_fixed_points_valid,
                    'basin_eligible': int(eligible),
                    'fixed_point_id': cluster_id,
                    'terminal_message_step': terminal_message_step,
                    'logical_correct': pcm_fixed_points_correct,
                    'logical_sector': pcm_fixed_points_sector,
                })

    pcm_fixed_points_restart_trials = pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_rows), BASIN_PATH)

arrival_summary = pcm_fixed_points_restart_trials.groupby(['shot_id', 'pcm_role'], as_index=False).agg(
    restarts=('restart', 'size'),
    fixed_point_arrivals=('convergence_class', lambda values: int((values == 'fixed_point').sum())),
    valid_basin_arrivals=('basin_eligible', 'sum'),
    valid_basin_arrival_rate=('basin_eligible', 'mean'),
)
eligible_restarts = pcm_fixed_points_restart_trials[pcm_fixed_points_restart_trials.basin_eligible == 1].copy()
basin_distribution = eligible_restarts.groupby(['shot_id', 'pcm_role', 'logical_sector'], as_index=False).size().rename(columns={'size': 'sector_frequency'})

if len(basin_distribution):
    distribution_summary = basin_distribution.groupby(['shot_id', 'pcm_role'], as_index=False).agg(restart_sector_count=('logical_sector', 'nunique'), basin_entropy=('sector_frequency', entropy_from_counts))
    state_distribution = eligible_restarts.groupby(['shot_id', 'pcm_role', 'fixed_point_id'], as_index=False).size()
    state_summary = state_distribution.groupby(['shot_id', 'pcm_role'], as_index=False).agg(fixed_point_count=('fixed_point_id', 'nunique'), basin_entropy=('size', entropy_from_counts))
    distribution_summary = distribution_summary.rename(columns={'basin_entropy': 'sector_entropy'}).merge(state_summary, on=['shot_id', 'pcm_role'])
    correct_frequency = eligible_restarts.groupby(['shot_id', 'pcm_role'], as_index=False).logical_correct.mean().rename(columns={'logical_correct': 'P_true'})
    distribution_summary = distribution_summary.merge(correct_frequency, on=['shot_id', 'pcm_role'])
else:
    distribution_summary = pd.DataFrame(columns=['shot_id', 'pcm_role', 'restart_sector_count', 'sector_entropy', 'fixed_point_count', 'basin_entropy', 'P_true'])

basin_summary = arrival_summary.merge(distribution_summary, on=['shot_id', 'pcm_role'], how='left')
basin_summary['restart_sector_count'] = basin_summary.restart_sector_count.fillna(0).astype(int)
basin_summary['fixed_point_count'] = basin_summary.fixed_point_count.fillna(0).astype(int)
pcm_fixed_points_save_csv(basin_summary, pcm_fixed_points_trace_dir / 'B2_B3_basin_summary.csv')
a1_wide = a1_case.pivot(index='shot_id', columns='view_kind', values=['correct_fraction', 'oracle_any_correct', 'sector_count'])
a1_wide.columns = [f'{metric}_{kind}' for metric, kind in a1_wide.columns]
a1_wide = a1_wide.reset_index()
b_analysis = pcm_fixed_points_margins[pcm_fixed_points_margins.cohort == 'gap'].merge(a1_wide, on='shot_id').merge(basin_summary[basin_summary.pcm_role == 'standard'].drop(columns='pcm_role'), on='shot_id')
development_margin = b_analysis[b_analysis.split == 'development'].normalized_logical_margin

if pcm_fixed_points_config['smoke_mode'] and development_margin.nunique() < 2:
    development_margin = b_analysis.normalized_logical_margin
assert len(development_margin)
quantiles = np.unique(development_margin.quantile(np.linspace(0, 1, pcm_fixed_points_config['logical_margin_quartiles'] + 1)).to_numpy())

if len(quantiles) < 2:
    quantiles = np.array([-np.inf, np.inf])
else:
    quantiles[0], quantiles[-1] = (-np.inf, np.inf)
b_analysis['margin_bin'] = pd.cut(b_analysis.normalized_logical_margin, quantiles, include_lowest=True, duplicates='drop').astype(str)
b_analysis['pcm_sensitive'] = b_analysis[[column for column in b_analysis if column.startswith('oracle_any_correct_')]].max(axis=1).astype(int)
b_analysis['basin_state'] = np.where(b_analysis.valid_basin_arrivals == 0, 'no_valid_fixed_point', np.where(b_analysis.fixed_point_count > 1, 'multi_basin', 'single_basin'))
b_analysis['multi_basin'] = np.where(b_analysis.valid_basin_arrivals == 0, np.nan, (b_analysis.fixed_point_count > 1).astype(float))
pcm_fixed_points_save_csv(b_analysis, pcm_fixed_points_trace_dir / 'B1_B2_analysis.csv')
correlation_rows = []

for response in ['correct_fraction_basis_replacement', 'correct_fraction_redundant_addition', 'basin_entropy', 'restart_sector_count', 'valid_basin_arrival_rate']:
    data = b_analysis[['normalized_logical_margin', response]].dropna()
    rho, pvalue = spearmanr(data.normalized_logical_margin, data[response]) if len(data) > 1 else (np.nan, np.nan)
    correlation_rows.append({
        'experiment': 'B1' if 'correct_fraction' in response else 'B2',
        'response': response,
        'cases': len(data),
        'spearman_rho': rho,
        'pvalue': pvalue,
    })

pcm_fixed_points_save_csv(pd.DataFrame(correlation_rows), pcm_fixed_points_data_paths.output('B1_B2_margin_correlations_csv'))
pcm_fixed_points_save_csv(
    b_analysis.groupby('margin_bin', as_index=False).agg(
        cases=('shot_id', 'size'),
        PCM_sensitive_rate=('pcm_sensitive', 'mean'),
        mean_basis_correct_fraction=('correct_fraction_basis_replacement', 'mean'),
        mean_redundant_correct_fraction=('correct_fraction_redundant_addition', 'mean'),
        mean_basin_entropy=('basin_entropy', 'mean'),
        mean_valid_basin_arrival_rate=('valid_basin_arrival_rate', 'mean'),
    ),
    pcm_fixed_points_data_paths.output('B1_B2_margin_bins_csv'),
)
pcm_fixed_points_save_csv(pcm_fixed_points_margins.groupby(['distance', 'cohort'], as_index=False).agg(cases=('shot_id', 'size'), median_normalized_margin=('normalized_logical_margin', 'median'), mean_normalized_margin=('normalized_logical_margin', 'mean')), pcm_fixed_points_data_paths.output('B1_cohort_margins_csv'))
b3_wide = basin_summary.pivot(index='shot_id', columns='pcm_role', values='P_true').reset_index()
b3 = b3_wide.dropna(subset=['standard', 'successful', 'failing']).copy() if {'standard', 'successful', 'failing'} <= set(b3_wide) else pd.DataFrame(columns=['shot_id', 'standard', 'successful', 'failing'])

if len(b3):
    b3['delta_P_true'] = b3.successful - b3.standard
    arrival_wide = basin_summary.pivot(index='shot_id', columns='pcm_role', values='valid_basin_arrival_rate')
    b3['delta_valid_basin_arrival_rate'] = b3.shot_id.map(arrival_wide.successful - arrival_wide.standard)
    standard_wrong = eligible_restarts[(eligible_restarts.pcm_role == 'standard') & (eligible_restarts.logical_correct == 0)].groupby('shot_id').logical_sector.agg(lambda x: x.value_counts().index[0])
    wrong_rows = []

    for pcm_fixed_points_shot_id in b3.shot_id:
        wrong = standard_wrong.get(pcm_fixed_points_shot_id, None)
        pcm_fixed_points_source = eligible_restarts[eligible_restarts.shot_id == pcm_fixed_points_shot_id]
        if wrong is None:
            wrong_rows.append(np.nan)
        else:
            probabilities = pcm_fixed_points_source.groupby('pcm_role').logical_sector.apply(lambda values: np.mean(values == wrong))
            wrong_rows.append(probabilities.get('successful', np.nan) - probabilities.get('standard', np.nan))

    b3['delta_P_wrong'] = wrong_rows

b3_columns = ['shot_id', 'standard', 'successful', 'failing', 'delta_P_true', 'delta_P_wrong', 'delta_valid_basin_arrival_rate']
pcm_fixed_points_save_csv(b3.reindex(columns=b3_columns), pcm_fixed_points_data_paths.output('B3_basin_probability_shift_csv'))
b4 = pcm_fixed_points_gap_cases[['shot_id', 'message_instability', 'sign_flip', 'extrinsic_conflict']].merge(b_analysis[['shot_id', 'normalized_logical_margin', 'pcm_sensitive', 'multi_basin', 'basin_state', 'margin_bin']], on='shot_id')
b4_rows = []

for metric in ['message_instability', 'sign_flip', 'extrinsic_conflict']:
    rho, pvalue = spearmanr(b4.normalized_logical_margin, b4[metric]) if len(b4) > 1 else (np.nan, np.nan)
    b4_rows.append({
        'metric': metric,
        'spearman_rho': rho,
        'pvalue': pvalue,
        'cases': len(b4),
    })

pcm_fixed_points_save_csv(pd.DataFrame(b4_rows), pcm_fixed_points_data_paths.output('B4_margin_oscillation_correlation_csv'))
pcm_fixed_points_save_csv(
    b4.groupby(['pcm_sensitive', 'basin_state', 'margin_bin'], as_index=False).agg(
        cases=('shot_id', 'size'),
        message_instability=('message_instability', 'mean'),
        sign_flip=('sign_flip', 'mean'),
        extrinsic_conflict=('extrinsic_conflict', 'mean'),
    ),
    pcm_fixed_points_data_paths.output('B4_grouped_oscillation_csv'),
)


## Part C — distinct targeted PCM ensembles and row coverage

Build distinct view ensembles and compare targeting scores and coverage as the view budget grows. Repeated copies of one representation should not be interpreted as new graph diversity.


In [ ]:
C3_ROW_PATH = pcm_fixed_points_trace_dir / 'C3_row_candidates.csv'
if pcm_fixed_points_stage_cached([C3_ROW_PATH]):
    row_trials = pcm_fixed_points_read_csv(C3_ROW_PATH)
else:
    pcm_fixed_points_rows = []
    for pcm_fixed_points_case_index, pcm_fixed_points_case in enumerate(tqdm(pcm_fixed_points_gap_cases.itertuples(index=False), total=len(pcm_fixed_points_gap_cases), desc='C3 row coverage', disable=pcm_fixed_points_config['print_level'] == 1)):
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_case.level, pcm_fixed_points_case.p)
        pcm_fixed_points_syndrome = pcm_fixed_points_bit_array(pcm_fixed_points_case.syndrome, pcm_fixed_points_context.H.shape[0])
        pcm_fixed_points_truth = pcm_fixed_points_bit_array(pcm_fixed_points_case.truth, pcm_fixed_points_context.L.shape[0])
        pcm_fixed_points_baseline = pcm_fixed_points_trace_bp(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_config['bp_standard_iterations'])
        pcm_fixed_points_endpoint = bp_endpoint(pcm_fixed_points_baseline)
        ambiguity, pcm_fixed_points_instability, conflict = variable_diagnostics(pcm_fixed_points_context, pcm_fixed_points_baseline, pcm_fixed_points_endpoint)
        pcm_fixed_points_successful = a1_views[(a1_views.shot_id == pcm_fixed_points_case.shot_id) & (a1_views.logical_correct == 1)].sort_values('candidate_id')
        pcm_fixed_points_repair_support = None if pcm_fixed_points_successful.empty else np.flatnonzero(pcm_fixed_points_bit_array(pcm_fixed_points_case.standard_bp_correction, pcm_fixed_points_context.H.shape[1]) ^ pcm_fixed_points_bit_array(pcm_fixed_points_successful.iloc[0].correction, pcm_fixed_points_context.H.shape[1]))

        for row_id, (coefficient, pcm_fixed_points_row) in enumerate(row_candidates(pcm_fixed_points_context)):
            pcm_fixed_points_support = np.flatnonzero(pcm_fixed_points_row)
            pcm_fixed_points_transform = np.vstack([np.eye(pcm_fixed_points_context.H.shape[0], dtype=np.uint8), coefficient])
            decoded = decode_view(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_truth, pcm_fixed_points_transform, f'{pcm_fixed_points_case.shot_id}:row:{row_id}', 'redundant_addition', 'C3_single_row', json.dumps(pcm_fixed_points_support.tolist()))
            decoded.update(
                shot_id=pcm_fixed_points_case.shot_id,
                distance=pcm_fixed_points_case.distance,
                p=pcm_fixed_points_case.p,
                split=pcm_fixed_points_case.split,
                row_id=row_id,
                coefficient_json=json.dumps(coefficient.tolist()),
                row_support=json.dumps(pcm_fixed_points_support.tolist()),
                ambiguity_coverage=float(ambiguity[pcm_fixed_points_support].sum()),
                instability_coverage=float(pcm_fixed_points_instability[pcm_fixed_points_support].sum()),
                conflict_coverage=float(conflict[pcm_fixed_points_support].sum()),
                oracle_repair_support_available=int(pcm_fixed_points_repair_support is not None),
                oracle_repair_support_coverage=np.nan if pcm_fixed_points_repair_support is None else len(set(pcm_fixed_points_support) & set(pcm_fixed_points_repair_support)),
            )
            pcm_fixed_points_rows.append(decoded)

    row_trials = pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_rows), C3_ROW_PATH)

def strategy_order(
    group,
    strategy,
    rng,
    mu=0.0,
):
    work = group.copy()
    if strategy == 'random':
        work['base_score'] = rng.random(len(work))
    elif strategy == 'hybrid':
        work['base_score'] = sum((work[name].rank(pct=True) for name in ['instability_coverage', 'ambiguity_coverage', 'conflict_coverage'])) / 3
    else:
        column = {
            'instability': 'instability_coverage',
            'ambiguity': 'ambiguity_coverage',
            'conflict': 'conflict_coverage',
        }[strategy]
        work['base_score'] = work[column]

    selected, remaining = ([], list(work.index))
    while remaining and len(selected) < max(pcm_fixed_points_config['pcm_budgets']):
        scored = []
        for index in remaining:
            support = set(json.loads(work.loc[index, 'row_support']))
            similarity = 0.0
            for chosen in selected:
                chosen_support = set(json.loads(work.loc[chosen, 'row_support']))
                union = support | chosen_support
                similarity = max(similarity, len(support & chosen_support) / len(union))

            scored.append((float(work.loc[index, 'base_score']) - mu * similarity, -int(work.loc[index, 'row_id']), index))

        best = max(scored)[2]
        selected.append(best)
        remaining.remove(best)

    assert len(selected) == max(pcm_fixed_points_config['pcm_budgets'])
    supports = [work.loc[index, 'row_support'] for index in selected]
    assert len(supports) == len(set(supports))

    return work.loc[selected].copy()

C12_PATH = pcm_fixed_points_trace_dir / 'C1_C2_targeted_candidates.csv'
if pcm_fixed_points_stage_cached([C12_PATH]):
    targeted_candidates = pcm_fixed_points_read_csv(C12_PATH)
else:
    pcm_fixed_points_rows = []
    cases = pcm_fixed_points_gap_cases.set_index('shot_id')
    grouped = row_trials.groupby('shot_id', sort=False)

    for pcm_fixed_points_case_index, (pcm_fixed_points_shot_id, pcm_fixed_points_group) in enumerate(tqdm(grouped, total=row_trials.shot_id.nunique(), desc='C1-C2 distinct ensembles', disable=pcm_fixed_points_config['print_level'] == 1)):
        pcm_fixed_points_case = cases.loc[pcm_fixed_points_shot_id]
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_case.level, pcm_fixed_points_case.p)
        pcm_fixed_points_syndrome = pcm_fixed_points_bit_array(pcm_fixed_points_case.syndrome, pcm_fixed_points_context.H.shape[0])
        pcm_fixed_points_truth = pcm_fixed_points_bit_array(pcm_fixed_points_case.truth, pcm_fixed_points_context.L.shape[0])

        for pcm_fixed_points_strategy in pcm_fixed_points_config['targeting_strategies']:
            strategy_seed = pcm_fixed_points_config['random_seed'] + pcm_fixed_points_case_index * 101 + pcm_fixed_points_config['targeting_strategies'].index(pcm_fixed_points_strategy)
            ordered = strategy_order(pcm_fixed_points_group, pcm_fixed_points_strategy, np.random.default_rng(strategy_seed), 0.0)
            for construction in [name for name in pcm_fixed_points_config['targeting_constructions'] if name != 'diversity_constrained']:
                coefficients = []
                for position, candidate in enumerate(ordered.itertuples(index=False), 1):
                    coefficient = np.asarray(json.loads(candidate.coefficient_json), dtype=np.uint8)
                    coefficients.append(coefficient)
                    extras = coefficient.reshape(1, -1) if construction == 'top_m_distinct' else np.asarray(coefficients)
                    pcm_fixed_points_transform = np.vstack([np.eye(pcm_fixed_points_context.H.shape[0], dtype=np.uint8), extras])
                    decoded = decode_view(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_truth, pcm_fixed_points_transform, f'{pcm_fixed_points_shot_id}:{pcm_fixed_points_strategy}:{construction}:{position}', 'redundant_addition', pcm_fixed_points_strategy, candidate.row_support)
                    decoded.update(
                        shot_id=pcm_fixed_points_shot_id,
                        distance=pcm_fixed_points_case.distance,
                        p=pcm_fixed_points_case.p,
                        split=pcm_fixed_points_case.split,
                        strategy=pcm_fixed_points_strategy,
                        construction=construction,
                        diversity_mu=0.0,
                        sequence_position=position,
                        row_id=candidate.row_id,
                    )
                    pcm_fixed_points_rows.append(decoded)

            for mu in pcm_fixed_points_config['diversity_mu_values'] if 'diversity_constrained' in pcm_fixed_points_config['targeting_constructions'] else []:
                diverse = strategy_order(pcm_fixed_points_group, pcm_fixed_points_strategy, np.random.default_rng(strategy_seed), mu)
                for position, candidate in enumerate(diverse.itertuples(index=False), 1):
                    coefficient = np.asarray(json.loads(candidate.coefficient_json), dtype=np.uint8)
                    pcm_fixed_points_transform = np.vstack([np.eye(pcm_fixed_points_context.H.shape[0], dtype=np.uint8), coefficient])
                    decoded = decode_view(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_truth, pcm_fixed_points_transform, f'{pcm_fixed_points_shot_id}:{pcm_fixed_points_strategy}:diversity:{mu:g}:{position}', 'redundant_addition', pcm_fixed_points_strategy, candidate.row_support)
                    decoded.update(
                        shot_id=pcm_fixed_points_shot_id,
                        distance=pcm_fixed_points_case.distance,
                        p=pcm_fixed_points_case.p,
                        split=pcm_fixed_points_case.split,
                        strategy=pcm_fixed_points_strategy,
                        construction='diversity_constrained',
                        diversity_mu=mu,
                        sequence_position=position,
                        row_id=candidate.row_id,
                    )
                    pcm_fixed_points_rows.append(decoded)

    targeted_candidates = pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_rows), C12_PATH)

group_keys = ['shot_id', 'strategy', 'construction', 'diversity_mu']
assert targeted_candidates.groupby(group_keys).candidate_id.nunique().eq(targeted_candidates.groupby(group_keys).size()).all()
assert targeted_candidates.groupby(group_keys).selected_row_support.nunique().ge(max(pcm_fixed_points_config['pcm_budgets'])).all()

def prefix_summary(candidates, group_columns, output):
    rows = []
    for keys, group in candidates.groupby(group_columns, sort=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        for budget in pcm_fixed_points_config['pcm_budgets']:
            prefix = group[group.sequence_position <= budget]
            valid_prefix = prefix[prefix.syndrome_consistent == 1]
            rows.append({
                **dict(zip(group_columns, keys)),
                'view_budget': budget,
                'oracle_generated': int(prefix.logical_correct.max()),
                'valid_candidate_count': len(valid_prefix),
                'sector_count': valid_prefix.logical_sector.nunique(),
                'pcm_sector_entropy': np.nan if valid_prefix.empty else entropy_from_counts(valid_prefix.logical_sector.value_counts().to_numpy()),
            })

    return pcm_fixed_points_save_csv(pd.DataFrame(rows), output)

c_prefix = prefix_summary(targeted_candidates, ['shot_id', 'distance', 'p', 'split', 'strategy', 'construction', 'diversity_mu'], pcm_fixed_points_trace_dir / 'C2_prefix_outcomes.csv')
pcm_fixed_points_save_csv(
    c_prefix.groupby(['strategy', 'construction', 'diversity_mu', 'view_budget'], as_index=False).agg(
        cases=('shot_id', 'size'),
        oracle_generation_rate=('oracle_generated', 'mean'),
        mean_valid_candidate_count=('valid_candidate_count', 'mean'),
        mean_sector_count=('sector_count', 'mean'),
        mean_pcm_sector_entropy=('pcm_sector_entropy', 'mean'),
    ),
    pcm_fixed_points_data_paths.output('C1_C2_distinct_targeted_scaling_csv'),
)
instability_source = pcm_fixed_points_gap_cases[pcm_fixed_points_gap_cases.split == 'development'].message_instability

if pcm_fixed_points_config['smoke_mode'] and instability_source.empty:
    instability_source = pcm_fixed_points_gap_cases.message_instability
assert len(instability_source)
c_stratified = c_prefix.merge(b_analysis[['shot_id', 'margin_bin', 'pcm_sensitive']], on='shot_id').merge(pcm_fixed_points_gap_cases[['shot_id', 'message_instability']], on='shot_id')
c_stratified['instability_group'] = np.where(c_stratified.message_instability >= instability_source.quantile(0.75), 'high', 'low')
c2_stratified_rows = []

for stratum in ['margin_bin', 'pcm_sensitive', 'instability_group']:
    aggregate = c_stratified.groupby([stratum, 'strategy', 'construction', 'diversity_mu', 'view_budget'], as_index=False).agg(cases=('shot_id', 'nunique'), oracle_generation_rate=('oracle_generated', 'mean'))
    aggregate['stratum'] = stratum
    aggregate = aggregate.rename(columns={stratum: 'stratum_value'})
    c2_stratified_rows.append(aggregate)

pcm_fixed_points_save_csv(pd.concat(c2_stratified_rows, ignore_index=True), pcm_fixed_points_data_paths.output('C2_stratified_generation_csv'))
c3_rows = []
for metric in ['ambiguity_coverage', 'instability_coverage', 'conflict_coverage', 'oracle_repair_support_coverage']:
    pcm_fixed_points_source = row_trials.dropna(subset=[metric]).copy()
    pcm_fixed_points_source['coverage_bin'] = pcm_fixed_points_source.groupby('shot_id')[metric].transform(lambda values: pd.qcut(values.rank(method='first'), 4, labels=False, duplicates='drop'))
    shot_level = pcm_fixed_points_source.groupby(['shot_id', 'coverage_bin'], as_index=False).agg(candidates=('candidate_id', 'size'), oracle_repair_probability=('logical_correct', 'mean'))

    for coverage_bin, pcm_fixed_points_group in shot_level.groupby('coverage_bin'):
        pcm_fixed_points_values = pcm_fixed_points_group.oracle_repair_probability.to_numpy(float)
        pcm_fixed_points_rng = np.random.default_rng(pcm_fixed_points_config['random_seed'] + len(c3_rows))
        pcm_fixed_points_indices = pcm_fixed_points_rng.integers(0, len(pcm_fixed_points_values), size=(pcm_fixed_points_config['bootstrap_repetitions'], len(pcm_fixed_points_values)))
        bootstrap = pcm_fixed_points_values[pcm_fixed_points_indices].mean(axis=1)
        c3_rows.append({
            'coverage_metric': metric,
            'coverage_bin': coverage_bin,
            'candidates': int(pcm_fixed_points_group.candidates.sum()),
            'shots': len(pcm_fixed_points_values),
            'oracle_repair_probability': float(pcm_fixed_points_values.mean()),
            'bootstrap_ci_low': float(np.quantile(bootstrap, 0.025)),
            'bootstrap_ci_high': float(np.quantile(bootstrap, 0.975)),
        })

c3_summary = pcm_fixed_points_save_csv(pd.DataFrame(c3_rows), pcm_fixed_points_data_paths.output('C3_row_coverage_repair_csv'))


## Part D — generation, sector diversity and non-oracle selectors

Measure whether a correct candidate exists, then compare non-oracle ways to choose among candidates. The generation oracle and practical selection success answer separate questions.


In [ ]:
d_parts = []
for pcm_fixed_points_kind, family in [('basis_replacement', 'basis_only'), ('redundant_addition', 'redundant_only')]:
    part = a1_views[a1_views.view_kind == pcm_fixed_points_kind].copy()
    part['family'] = family
    part['sequence_position'] = part.view_id
    d_parts.append(part)

mixed_rows = []
for pcm_fixed_points_shot_id, pcm_fixed_points_group in a1_views.groupby('shot_id', sort=False):
    basis = pcm_fixed_points_group[pcm_fixed_points_group.view_kind == 'basis_replacement'].sort_values('view_id').copy()
    basis['mixed_order'] = 2 * basis.view_id - 1
    redundant = pcm_fixed_points_group[pcm_fixed_points_group.view_kind == 'redundant_addition'].sort_values('view_id').copy()
    redundant['mixed_order'] = 2 * redundant.view_id
    mixed = pd.concat([basis, redundant]).sort_values('mixed_order')
    mixed['family'] = 'random_mixed'
    mixed['sequence_position'] = np.arange(1, len(mixed) + 1)
    mixed_rows.append(mixed)

d_parts.append(pd.concat(mixed_rows, ignore_index=True))
target_part = targeted_candidates[(targeted_candidates.construction == 'top_m_distinct') & (targeted_candidates.diversity_mu == 0)].copy()
target_part['family'] = target_part.strategy + '_targeted'
d_parts.append(target_part)
d_candidates = pd.concat(d_parts, ignore_index=True, sort=False)
d_candidates = d_candidates[d_candidates.sequence_position <= max(pcm_fixed_points_config['pcm_budgets'])].copy()
d_prefix = prefix_summary(d_candidates, ['shot_id', 'distance', 'p', 'split', 'family'], pcm_fixed_points_trace_dir / 'D1_D2_candidate_prefixes.csv')
pcm_fixed_points_save_csv(
    d_prefix.groupby(['family', 'view_budget'], as_index=False).agg(
        cases=('shot_id', 'size'),
        oracle_generation_rate=('oracle_generated', 'mean'),
        mean_valid_candidate_count=('valid_candidate_count', 'mean'),
        mean_sector_count=('sector_count', 'mean'),
        mean_pcm_sector_entropy=('pcm_sector_entropy', 'mean'),
    ),
    pcm_fixed_points_data_paths.output('D1_D2_generation_diversity_csv'),
)
d2_margin = d_prefix.merge(pcm_fixed_points_margins[['shot_id', 'normalized_logical_margin']], on='shot_id', how='left')
d2_rows = []

for (family, budget), pcm_fixed_points_group in d2_margin.groupby(['family', 'view_budget']):
    data = pcm_fixed_points_group.dropna(subset=['normalized_logical_margin', 'pcm_sector_entropy'])
    rho, pvalue = spearmanr(data.normalized_logical_margin, data.pcm_sector_entropy) if len(data) > 1 else (np.nan, np.nan)
    d2_rows.append({
        'family': family,
        'view_budget': budget,
        'cases': data.shot_id.nunique(),
        'spearman_rho': rho,
        'pvalue': pvalue,
    })

pcm_fixed_points_save_csv(pd.DataFrame(d2_rows), pcm_fixed_points_data_paths.output('D2_margin_sector_entropy_csv'))

def selected_candidate(group, selector, parameter=None):
    valid = group[group.syndrome_consistent == 1].copy()
    if valid.empty:
        return None
    scale = max(1e-12, float(valid.prior_cost.abs().median()))
    valid['normalized_cost'] = valid.prior_cost / scale

    if selector == 'minimum_prior_cost':
        return valid.sort_values(['prior_cost', 'candidate_id']).iloc[0]
    if selector == 'majority':
        table = valid.groupby('logical_sector').agg(frequency=('candidate_id', 'size'), min_cost=('prior_cost', 'min')).reset_index()
        sector = table.sort_values(['frequency', 'min_cost', 'logical_sector'], ascending=[False, True, True]).iloc[0].logical_sector
    elif selector == 'frequency_cost':
        table = valid.groupby('logical_sector').agg(frequency=('candidate_id', 'size'), min_cost=('normalized_cost', 'min')).reset_index()
        table['score'] = -table.min_cost + float(parameter) * np.log(table.frequency / len(valid) + pcm_fixed_points_config['selector_confidence_epsilon'])
        sector = table.sort_values(['score', 'logical_sector'], ascending=[False, True]).iloc[0].logical_sector
    elif selector == 'weighted_consensus':
        valid['weight'] = 1 / (valid.view_ambiguity + pcm_fixed_points_config['selector_confidence_epsilon'])
        table = valid.groupby('logical_sector').weight.sum().reset_index(name='score')
        sector = table.sort_values(['score', 'logical_sector'], ascending=[False, True]).iloc[0].logical_sector
    else:
        assert selector == 'approximate_mass'
        table = valid.groupby('logical_sector').normalized_cost.apply(lambda values: logsumexp(-values.to_numpy() / float(parameter))).reset_index(name='score')
        sector = table.sort_values(['score', 'logical_sector'], ascending=[False, True]).iloc[0].logical_sector

    return valid[valid.logical_sector == sector].sort_values(['prior_cost', 'candidate_id']).iloc[0]

selectors = {
    'minimum_prior_cost': [None],
    'majority': [None],
    'frequency_cost': pcm_fixed_points_config['selector_eta_values'],
    'weighted_consensus': [None],
    'approximate_mass': pcm_fixed_points_config['selector_temperature_values'],
}
tuning_rows = []
development_candidates = d_candidates[d_candidates.split == 'development']

if pcm_fixed_points_config['smoke_mode'] and development_candidates.empty:
    development_candidates = d_candidates
assert len(development_candidates)

for selector, parameters in selectors.items():
    for parameter in parameters:
        outcomes = []
        for pcm_fixed_points_, pcm_fixed_points_group in development_candidates.groupby(['shot_id', 'family']):
            pcm_fixed_points_chosen = selected_candidate(pcm_fixed_points_group, selector, parameter)
            outcomes.append(0 if pcm_fixed_points_chosen is None else int(pcm_fixed_points_chosen.logical_correct))

        tuning_rows.append({
            'selector': selector,
            'parameter': np.nan if parameter is None else parameter,
            'development_cases': len(outcomes),
            'development_accuracy': np.mean(outcomes),
        })

selector_tuning = pd.DataFrame(tuning_rows)
best_parameters = {}
for selector, pcm_fixed_points_group in selector_tuning.groupby('selector'):
    best = pcm_fixed_points_group.sort_values(['development_accuracy', 'parameter'], ascending=[False, True], na_position='first').iloc[0]
    best_parameters[selector] = None if pd.isna(best.parameter) else float(best.parameter)

pcm_fixed_points_save_csv(selector_tuning, pcm_fixed_points_data_paths.output('D3_selector_tuning_csv'))
selector_rows = []
for (pcm_fixed_points_shot_id, family), pcm_fixed_points_group in tqdm(d_candidates.groupby(['shot_id', 'family'], sort=False), desc='D3-D5 selectors', disable=pcm_fixed_points_config['print_level'] == 1):
    metadata = pcm_fixed_points_group.iloc[0]
    for budget in pcm_fixed_points_config['pcm_budgets']:
        prefix = pcm_fixed_points_group[pcm_fixed_points_group.sequence_position <= budget]
        pcm_fixed_points_oracle = int(prefix.logical_correct.max())
        for selector, parameter in best_parameters.items():
            pcm_fixed_points_chosen = selected_candidate(prefix, selector, parameter)
            selected_success = 0 if pcm_fixed_points_chosen is None else int(pcm_fixed_points_chosen.logical_correct)
            selector_rows.append({
                'shot_id': pcm_fixed_points_shot_id,
                'distance': metadata.distance,
                'p': metadata.p,
                'split': metadata.split,
                'family': family,
                'view_budget': budget,
                'selector': selector,
                'parameter': parameter,
                'oracle_generated': pcm_fixed_points_oracle,
                'selected_success': selected_success,
                'generation_selection_failure': int(pcm_fixed_points_oracle and (not selected_success)),
            })

selector_trials = pcm_fixed_points_save_csv(pd.DataFrame(selector_rows), pcm_fixed_points_trace_dir / 'D3_D5_selector_trials.csv')
development_selectors = selector_trials[selector_trials.split == 'development']
if pcm_fixed_points_config['smoke_mode'] and development_selectors.empty:
    development_selectors = selector_trials
best_selector_row = development_selectors.groupby('selector', as_index=False).selected_success.mean().sort_values(['selected_success', 'selector'], ascending=[False, True]).iloc[0]
BEST_SELECTOR = best_selector_row.selector
BEST_PARAMETER = best_parameters[BEST_SELECTOR]
pcm_fixed_points_save_csv(pd.DataFrame([{
    'selector': BEST_SELECTOR,
    'parameter': BEST_PARAMETER,
    'selection_split': 'development',
}]), pcm_fixed_points_data_paths.output('D3_frozen_selector_csv'))
pcm_fixed_points_save_csv(
    selector_trials.groupby(['family', 'view_budget', 'selector'], as_index=False).agg(
        cases=('shot_id', 'size'),
        oracle_generation_rate=('oracle_generated', 'mean'),
        selected_success_rate=('selected_success', 'mean'),
        generation_selection_gap=('generation_selection_failure', 'mean'),
    ),
    pcm_fixed_points_data_paths.output('D5_generation_selection_gap_csv'),
)
d4 = selector_trials.merge(b_analysis[['shot_id', 'margin_bin']], on='shot_id', how='left')
pcm_fixed_points_save_csv(d4.groupby(['family', 'view_budget', 'selector', 'margin_bin'], as_index=False, dropna=False).agg(cases=('shot_id', 'size'), selector_accuracy=('selected_success', 'mean')), pcm_fixed_points_data_paths.output('D4_selector_margin_csv'))


## Part E — frozen PCM repair, OSD1 handoff and unbiased final LER

Run the frozen PCM-repair and OSD1 variants on the unbiased final population. This is the end-to-end test of whether candidate diversity and selection produce useful LER after regression costs.


In [ ]:
def make_final_candidates(
    context,
    syndrome,
    truth,
    baseline,
    family,
    count,
    seed,
):
    rng = np.random.default_rng(seed)
    records = []
    if family == 'same_PCM_restart':
        for position in range(1, count + 1):
            trace = pcm_fixed_points_trace_bp(context, syndrome, pcm_fixed_points_config['bp_standard_iterations'], rng=rng, initial_message_sigma=pcm_fixed_points_config['restart_message_sigma'])
            endpoint = bp_endpoint(trace)
            correction = trace.hard[endpoint]
            valid, correct, _, sector = pcm_fixed_points_outcome(context, correction, syndrome, truth)
            metrics = pcm_fixed_points_trace_metrics(context, trace, endpoint)

            records.append({
                'candidate_id': f'restart:{position}',
                'sequence_position': position,
                'view_kind': 'same_pcm_restart',
                'pcm_source': 'same_PCM_restart',
                'selected_row_support': '',
                'syndrome_consistent': valid,
                'logical_correct': correct,
                'logical_sector': sector,
                'correction': pcm_fixed_points_bits(correction),
                'llr_json': json.dumps(trace.llr[endpoint].tolist()),
                'prior_cost': pcm_fixed_points_prior_cost(context, correction),
                'bp_confidence': metrics['posterior_abs_mean'],
                'view_ambiguity': float(np.mean(1 / (np.abs(trace.llr[endpoint]) + pcm_fixed_points_config['selector_confidence_epsilon']))),
                'runtime': float(trace.runtime[endpoint]),
                'iterations': endpoint,
                **metrics,
            })

        return pd.DataFrame(records)

    if family == 'basis_ensemble':
        transforms = random_basis_transforms(context, count, rng)
    elif family == 'redundant_ensemble':
        transforms = random_redundant_transforms(context, count, rng)
    elif family == 'random_mixed':
        basis = random_basis_transforms(context, (count + 1) // 2, rng)
        redundant = random_redundant_transforms(context, count // 2, rng)
        transforms = []

        for index in range(count):
            transforms.append(basis[index // 2] if index % 2 == 0 else redundant[index // 2])
    else:
        assert family in {'ambiguity_targeted', 'instability_targeted'}
        endpoint = bp_endpoint(baseline)
        ambiguity, instability, _ = variable_diagnostics(context, baseline, endpoint)
        candidates = row_candidates(context)
        values = ambiguity if family == 'ambiguity_targeted' else instability
        scored = sorted(candidates, key=lambda item: (-float(values[np.flatnonzero(item[1])].sum()), item[1].tobytes()))[:count]
        transforms = [np.vstack([np.eye(context.H.shape[0], dtype=np.uint8), coefficient]) for coefficient, _ in scored]

    assert len(transforms) == count
    for position, transform in enumerate(transforms, 1):
        record = decode_view(context, syndrome, truth, transform, f'{family}:{position}', 'basis_replacement' if transform.shape[0] == context.H.shape[0] else 'redundant_addition', family)
        record['sequence_position'] = position
        records.append(record)

    return pd.DataFrame(records)

E1_PATH = pcm_fixed_points_trace_dir / 'E1_pcm_ensemble_trials.csv'
E1_CANDIDATE_PATH = pcm_fixed_points_trace_dir / 'E1_candidate_details.csv'
E1_COMPLETE_PATH = pcm_fixed_points_trace_dir / 'E1_complete.json'
E1_CANDIDATE_COLUMNS = ['shot_id', 'distance', 'p', 'split', 'population_role', 'family', 'candidate_id', 'sequence_position', 'logical_sector', 'prior_cost', 'bp_confidence', 'pcm_source', 'view_kind', 'syndrome_consistent', 'logical_correct', 'view_ambiguity', 'runtime', 'iterations']
e4_population = unbiased.copy()

if pcm_fixed_points_config['final_case_limit_per_point'] > 0:
    e4_population = e4_population.groupby(['distance', 'p', 'split'], sort=False).head(pcm_fixed_points_config['final_case_limit_per_point'])
e4_population_ids = set(e4_population.shot_id)
e_population = pd.concat([e4_population, pcm_fixed_points_gap_cases], ignore_index=True).drop_duplicates('shot_id')
E1_FAMILIES = ['same_PCM_restart', 'basis_ensemble', 'redundant_ensemble', 'random_mixed', 'ambiguity_targeted', 'instability_targeted']
E1_TRIAL_FAMILIES = ['standard_BP', *E1_FAMILIES]

def validate_e1_progress(trials, candidates, require_complete):
    assert not trials.duplicated(['shot_id', 'family']).any(), 'duplicate E1 trial rows'
    assert not candidates.duplicated(['shot_id', 'family', 'candidate_id']).any(), 'duplicate E1 candidates'
    trial_families = trials.groupby('shot_id').family.agg(set)
    candidate_families = candidates.groupby('shot_id').family.agg(set)
    expected_trials, expected_candidates = (set(E1_TRIAL_FAMILIES), set(E1_FAMILIES))
    assert trial_families.map(lambda values: values == expected_trials).all(), 'incomplete E1 trial family block'
    assert candidate_families.map(lambda values: values == expected_candidates).all(), 'incomplete E1 candidate family block'
    candidate_counts = candidates.groupby(['shot_id', 'family']).size()
    assert candidate_counts.eq(pcm_fixed_points_config['final_pcm_views']).all(), 'incomplete E1 candidate view block'
    completed = set(trial_families.index)
    assert completed == set(candidate_families.index), 'E1 trial/candidate shot mismatch'
    population_ids = set(e_population.shot_id)
    assert completed <= population_ids, 'E1 cache contains unknown shots'

    if require_complete:
        assert completed == population_ids, 'E1 completion marker does not cover the full population'

    return completed

data_present = [E1_PATH.is_file(), E1_CANDIDATE_PATH.is_file()]
marker_present = E1_COMPLETE_PATH.is_file()
if not pcm_fixed_points_config['reuse_existing_data']:
    for pcm_fixed_points_path in [E1_PATH, E1_CANDIDATE_PATH, E1_COMPLETE_PATH]:
        if pcm_fixed_points_path.is_file():
            pcm_fixed_points_path.unlink()

    data_present = [False, False]
    marker_present = False

assert all(data_present) or not any(data_present), 'E1 resume requires both trace CSV files'
assert not marker_present or all(data_present), 'E1 completion marker requires both trace CSV files'
if all(data_present):
    e1_trials = pcm_fixed_points_read_csv(E1_PATH)
    e1_candidates = pcm_fixed_points_read_csv(E1_CANDIDATE_PATH)
    completed_e1_shots = validate_e1_progress(e1_trials, e1_candidates, require_complete=marker_present)
else:
    completed_e1_shots = set()

if marker_present:
    completion = json.loads(E1_COMPLETE_PATH.read_text())
    assert completion['complete'] is True
    pcm_fixed_points_log(2, {'stage': 'E1', 'cache': 'complete', 'shots': len(completed_e1_shots)})
else:
    pcm_fixed_points_log(2, {
        'stage': 'E1',
        'cache': 'resume' if completed_e1_shots else 'new',
        'completed_shots': len(completed_e1_shots),
        'total_shots': len(e_population),
    })
    trial_rows, candidate_rows = ([], [])

    def flush_e1(rows, path, columns=None):
        if not rows:
            return
        frame = pd.DataFrame(rows, columns=columns)

        frame.to_csv(path, mode='a', header=not path.is_file(), index=False)
        rows.clear()

    new_shots = 0
    for pcm_fixed_points_case_index, pcm_fixed_points_case in enumerate(tqdm(e_population.itertuples(index=False), total=len(e_population), desc=f'E1 ensembles ({len(completed_e1_shots)} cached)', disable=pcm_fixed_points_config['print_level'] == 1)):
        if pcm_fixed_points_case.shot_id in completed_e1_shots:
            continue
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_case.level, pcm_fixed_points_case.p)
        pcm_fixed_points_syndrome = pcm_fixed_points_bit_array(pcm_fixed_points_case.syndrome, pcm_fixed_points_context.H.shape[0])
        pcm_fixed_points_truth = pcm_fixed_points_bit_array(pcm_fixed_points_case.truth, pcm_fixed_points_context.L.shape[0])
        pcm_fixed_points_baseline = pcm_fixed_points_trace_bp(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_config['bp_standard_iterations'])
        bp_end = bp_endpoint(pcm_fixed_points_baseline)
        base_valid, base_correct, pcm_fixed_points_, base_sector = pcm_fixed_points_outcome(pcm_fixed_points_context, pcm_fixed_points_baseline.hard[bp_end], pcm_fixed_points_syndrome, pcm_fixed_points_truth)

        trial_rows.append({
            'shot_id': pcm_fixed_points_case.shot_id,
            'distance': pcm_fixed_points_case.distance,
            'p': pcm_fixed_points_case.p,
            'split': pcm_fixed_points_case.split,
            'population_role': pcm_fixed_points_case.population_role,
            'family': 'standard_BP',
            'logical_correct': base_correct,
            'syndrome_consistent': base_valid,
            'logical_sector': base_sector,
            'selected_correction': pcm_fixed_points_bits(pcm_fixed_points_baseline.hard[bp_end]),
            'selected_llr_json': json.dumps(pcm_fixed_points_baseline.llr[bp_end].tolist()),
            'bp_runs': 1,
            'pcm_views': 1,
            'iterations': bp_end,
            'runtime': float(pcm_fixed_points_baseline.runtime[bp_end]),
        })

        for family_index, family in enumerate(E1_FAMILIES):
            family_started = perf_counter()
            pcm_fixed_points_candidates = make_final_candidates(pcm_fixed_points_context, pcm_fixed_points_syndrome, pcm_fixed_points_truth, pcm_fixed_points_baseline, family, pcm_fixed_points_config['final_pcm_views'], pcm_fixed_points_config['random_seed'] + pcm_fixed_points_case_index * 101 + family_index)
            pcm_fixed_points_candidates['shot_id'] = pcm_fixed_points_case.shot_id
            pcm_fixed_points_candidates['distance'] = pcm_fixed_points_case.distance
            pcm_fixed_points_candidates['p'] = pcm_fixed_points_case.p
            pcm_fixed_points_candidates['split'] = pcm_fixed_points_case.split
            pcm_fixed_points_candidates['population_role'] = pcm_fixed_points_case.population_role
            pcm_fixed_points_candidates['family'] = family
            candidate_rows.extend(pcm_fixed_points_candidates[E1_CANDIDATE_COLUMNS].to_dict('records'))
            pcm_fixed_points_chosen = selected_candidate(pcm_fixed_points_candidates, BEST_SELECTOR, BEST_PARAMETER)

            if pcm_fixed_points_chosen is None:
                pcm_fixed_points_correction, pcm_fixed_points_llr, pcm_fixed_points_valid, pcm_fixed_points_correct, pcm_fixed_points_sector = (pcm_fixed_points_baseline.hard[bp_end], pcm_fixed_points_baseline.llr[bp_end], base_valid, base_correct, base_sector)
            else:
                pcm_fixed_points_correction = pcm_fixed_points_bit_array(pcm_fixed_points_chosen.correction, pcm_fixed_points_context.H.shape[1])
                pcm_fixed_points_llr = np.asarray(json.loads(pcm_fixed_points_chosen.llr_json))

                pcm_fixed_points_valid, pcm_fixed_points_correct, pcm_fixed_points_sector = (int(pcm_fixed_points_chosen.syndrome_consistent), int(pcm_fixed_points_chosen.logical_correct), pcm_fixed_points_chosen.logical_sector)

            pcm_fixed_points_runtime = float(pcm_fixed_points_baseline.runtime[bp_end] + perf_counter() - family_started)
            pcm_fixed_points_iterations = int(bp_end + pcm_fixed_points_candidates.iterations.sum())
            trial_rows.append({
                'shot_id': pcm_fixed_points_case.shot_id,
                'distance': pcm_fixed_points_case.distance,
                'p': pcm_fixed_points_case.p,
                'split': pcm_fixed_points_case.split,
                'population_role': pcm_fixed_points_case.population_role,
                'family': family,
                'logical_correct': pcm_fixed_points_correct,
                'syndrome_consistent': pcm_fixed_points_valid,
                'logical_sector': pcm_fixed_points_sector,
                'selected_correction': pcm_fixed_points_bits(pcm_fixed_points_correction),
                'selected_llr_json': json.dumps(pcm_fixed_points_llr.tolist()),
                'bp_runs': 1 + len(pcm_fixed_points_candidates),
                'pcm_views': 1 + len(pcm_fixed_points_candidates),
                'iterations': pcm_fixed_points_iterations,
                'runtime': pcm_fixed_points_runtime,
            })

        new_shots += 1
        if new_shots % pcm_fixed_points_config['e1_stream_chunk_shots'] == 0:
            flush_e1(trial_rows, E1_PATH)
            flush_e1(candidate_rows, E1_CANDIDATE_PATH, E1_CANDIDATE_COLUMNS)

    flush_e1(trial_rows, E1_PATH)
    flush_e1(candidate_rows, E1_CANDIDATE_PATH, E1_CANDIDATE_COLUMNS)
    e1_trials = pcm_fixed_points_read_csv(E1_PATH)
    e1_candidates = pcm_fixed_points_read_csv(E1_CANDIDATE_PATH)
    completed_e1_shots = validate_e1_progress(e1_trials, e1_candidates, require_complete=True)

    E1_COMPLETE_PATH.write_text(json.dumps({'complete': True}))

development_e1 = e1_trials[(e1_trials.split == 'development') & (e1_trials.population_role == 'unbiased') & e1_trials.distance.isin(pcm_fixed_points_primary_distances)]
assert len(development_e1)
family_policy_table = development_e1.groupby('family', as_index=False).logical_correct.mean()
pcm_families = family_policy_table[~family_policy_table.family.isin(['standard_BP', 'same_PCM_restart'])]
best_family = pcm_families.sort_values(['logical_correct', 'family'], ascending=[False, True]).iloc[0].family
targeted_policy = pcm_families[pcm_families.family.isin(['ambiguity_targeted', 'instability_targeted'])].sort_values(['logical_correct', 'family'], ascending=[False, True]).iloc[0].family
pcm_fixed_points_policy = pd.DataFrame([{
    'best_selector': BEST_SELECTOR,
    'selector_parameter': BEST_PARAMETER,
    'best_pcm_family': best_family,
    'best_targeted_family': targeted_policy,
    'view_budget': pcm_fixed_points_config['final_pcm_views'],
    'selection_split': 'development',
}])
pcm_fixed_points_save_csv(pcm_fixed_points_policy, pcm_fixed_points_data_paths.output('E_frozen_policy_csv'))
standard_lookup = pcm_fixed_points_population.set_index('shot_id')
e1_summary_rows = []

for (pcm_fixed_points_population_role, pcm_fixed_points_split_name, pcm_fixed_points_distance, pcm_fixed_points_p, family), pcm_fixed_points_group in e1_trials.groupby(['population_role', 'split', 'distance', 'p', 'family']):
    pcm_fixed_points_reference = standard_lookup.loc[pcm_fixed_points_group.shot_id]
    pcm_fixed_points_gap_mask = (pcm_fixed_points_reference.standard_correct.to_numpy() == 0) & (pcm_fixed_points_reference.tesseract_correct.to_numpy() == 1)
    baseline_correct = pcm_fixed_points_reference.standard_correct.to_numpy() == 1
    e1_summary_rows.append({
        'population_role': pcm_fixed_points_population_role,
        'split': pcm_fixed_points_split_name,
        'distance': pcm_fixed_points_distance,
        'p': pcm_fixed_points_p,
        'family': family,
        'shots': len(pcm_fixed_points_group),
        'LER': 1 - pcm_fixed_points_group.logical_correct.mean(),
        'conditional_gap_repair': np.nan if not pcm_fixed_points_gap_mask.any() else pcm_fixed_points_group.logical_correct.to_numpy()[pcm_fixed_points_gap_mask].mean(),
        'regression': np.nan if not baseline_correct.any() else 1 - pcm_fixed_points_group.logical_correct.to_numpy()[baseline_correct].mean(),
        'mean_bp_runs': pcm_fixed_points_group.bp_runs.mean(),
        'mean_pcm_views': pcm_fixed_points_group.pcm_views.mean(),
        'mean_runtime': pcm_fixed_points_group.runtime.mean(),
    })

pcm_fixed_points_save_csv(pd.DataFrame(e1_summary_rows), pcm_fixed_points_data_paths.output('E1_pcm_ensemble_summary_csv'))
E2_PATH = pcm_fixed_points_trace_dir / 'E2_decoder_trials.csv'
Q_PATH = pcm_fixed_points_trace_dir / 'E2_qmin_before_after.csv'

if pcm_fixed_points_stage_cached([E2_PATH, Q_PATH]):
    e2_trials = pcm_fixed_points_read_csv(E2_PATH)
    q_trials = pcm_fixed_points_read_csv(Q_PATH)
else:
    pcm_fixed_points_rows, q_rows = ([], [])
    standard_rows = e1_trials[e1_trials.family == 'standard_BP']
    assert not e1_trials.duplicated(['shot_id', 'family']).any()
    best_lookup = e1_trials[e1_trials.family == best_family].set_index('shot_id', verify_integrity=True)
    targeted_lookup = e1_trials[e1_trials.family == targeted_policy].set_index('shot_id', verify_integrity=True)

    for pcm_fixed_points_case in tqdm(standard_rows.itertuples(index=False), total=len(standard_rows), desc='E2-E4 decoder trials', disable=pcm_fixed_points_config['print_level'] == 1):
        pcm_fixed_points_source = standard_lookup.loc[pcm_fixed_points_case.shot_id]
        pcm_fixed_points_context = pcm_fixed_points_context_for(pcm_fixed_points_source.level, pcm_fixed_points_source.p)
        pcm_fixed_points_syndrome = pcm_fixed_points_bit_array(pcm_fixed_points_source.syndrome, pcm_fixed_points_context.H.shape[0])
        pcm_fixed_points_truth = pcm_fixed_points_bit_array(pcm_fixed_points_source.truth, pcm_fixed_points_context.L.shape[0])

        bp_llr = np.asarray(json.loads(pcm_fixed_points_source.standard_bp_llr_json))
        best = best_lookup.loc[pcm_fixed_points_case.shot_id]
        targeted = targeted_lookup.loc[pcm_fixed_points_case.shot_id]

        best_llr = np.asarray(json.loads(best.selected_llr_json))
        endpoint_osd1, endpoint_time = pcm_fixed_points_osd_decode(pcm_fixed_points_context, pcm_fixed_points_syndrome, bp_llr, 1)
        pcm_osd1, pcm_time = pcm_fixed_points_osd_decode(pcm_fixed_points_context, pcm_fixed_points_syndrome, best_llr, 1)
        full, full_time = pcm_fixed_points_osd_decode(pcm_fixed_points_context, pcm_fixed_points_syndrome, bp_llr, pcm_fixed_points_full_osd_order(pcm_fixed_points_context))
        pcm_fixed_points_methods = [('standard_BPOSD', int(pcm_fixed_points_source.standard_correct), int(pcm_fixed_points_source.standard_valid), pcm_fixed_points_source.standard_sector, float(pcm_fixed_points_source.standard_runtime), 1, 1, int(pcm_fixed_points_case.iterations), int(pcm_fixed_points_source.standard_osd_used), 0), ('forced_endpoint_OSD1', pcm_fixed_points_outcome(pcm_fixed_points_context, endpoint_osd1, pcm_fixed_points_syndrome, pcm_fixed_points_truth)[1], 1, pcm_fixed_points_logical_label(pcm_fixed_points_context, endpoint_osd1), pcm_fixed_points_source.standard_bp_runtime + endpoint_time, 1, 1, int(pcm_fixed_points_case.iterations), 0, 1), ('best_PCM_only_BP', int(best.logical_correct), int(best.syndrome_consistent), best.logical_sector, best.runtime, best.bp_runs, best.pcm_views, int(best.iterations), 0, 0), ('best_targeted_PCM_BP', int(targeted.logical_correct), int(targeted.syndrome_consistent), targeted.logical_sector, targeted.runtime, targeted.bp_runs, targeted.pcm_views, int(targeted.iterations), 0, 0), ('best_PCM_plus_OSD1', pcm_fixed_points_outcome(pcm_fixed_points_context, pcm_osd1, pcm_fixed_points_syndrome, pcm_fixed_points_truth)[1], 1, pcm_fixed_points_logical_label(pcm_fixed_points_context, pcm_osd1), best.runtime + pcm_time, best.bp_runs, best.pcm_views, int(best.iterations), 0, 1), ('forced_full_OSD', pcm_fixed_points_outcome(pcm_fixed_points_context, full, pcm_fixed_points_syndrome, pcm_fixed_points_truth)[1], 1, pcm_fixed_points_logical_label(pcm_fixed_points_context, full), pcm_fixed_points_source.standard_bp_runtime + full_time, 1, 1, int(pcm_fixed_points_case.iterations), 1, 0), ('Tesseract', int(pcm_fixed_points_source.tesseract_correct), np.nan, pcm_fixed_points_source.tesseract_sector, pcm_fixed_points_source.tesseract_runtime, 0, 0, 0, 0, 0)]

        for pcm_fixed_points_method, pcm_fixed_points_correct, pcm_fixed_points_valid, pcm_fixed_points_sector, pcm_fixed_points_runtime, runs, pcm_fixed_points_views, pcm_fixed_points_iterations, full_used, osd1_used in pcm_fixed_points_methods:
            pcm_fixed_points_rows.append({
                'shot_id': pcm_fixed_points_case.shot_id,
                'distance': pcm_fixed_points_case.distance,
                'p': pcm_fixed_points_case.p,
                'split': pcm_fixed_points_case.split,
                'population_role': pcm_fixed_points_case.population_role,
                'method': pcm_fixed_points_method,
                'logical_correct': pcm_fixed_points_correct,
                'syndrome_consistent': pcm_fixed_points_valid,
                'logical_sector': pcm_fixed_points_sector,
                'runtime': pcm_fixed_points_runtime,
                'bp_runs': runs,
                'pcm_views': pcm_fixed_points_views,
                'fraction_full_OSD': full_used,
                'fraction_OSD1': osd1_used,
                'iterations': pcm_fixed_points_iterations,
                'standard_correct': int(pcm_fixed_points_source.standard_correct),
                'tesseract_correct': int(pcm_fixed_points_source.tesseract_correct),
            })

        if int(pcm_fixed_points_source.gap):
            before, after = ('unrepaired', 'unrepaired')
            pcm_fixed_points_orders = list(dict.fromkeys(pcm_fixed_points_config['osd_orders'] + [pcm_fixed_points_full_osd_order(pcm_fixed_points_context)]))
            for pcm_fixed_points_order in pcm_fixed_points_orders:
                if before == 'unrepaired' and pcm_fixed_points_outcome(pcm_fixed_points_context, pcm_fixed_points_osd_decode(pcm_fixed_points_context, pcm_fixed_points_syndrome, bp_llr, pcm_fixed_points_order)[0], pcm_fixed_points_syndrome, pcm_fixed_points_truth)[1]:
                    before = str(pcm_fixed_points_order)
                if after == 'unrepaired' and pcm_fixed_points_outcome(pcm_fixed_points_context, pcm_fixed_points_osd_decode(pcm_fixed_points_context, pcm_fixed_points_syndrome, best_llr, pcm_fixed_points_order)[0], pcm_fixed_points_syndrome, pcm_fixed_points_truth)[1]:
                    after = str(pcm_fixed_points_order)

            before_number = np.inf if before == 'unrepaired' else int(before)
            after_number = np.inf if after == 'unrepaired' else int(after)
            q_rows.append({
                'shot_id': pcm_fixed_points_case.shot_id,
                'distance': pcm_fixed_points_case.distance,
                'p': pcm_fixed_points_case.p,
                'split': pcm_fixed_points_case.split,
                'q_min_before': before,
                'q_min_after': after,
                'q_min_improved': int(after_number < before_number),
            })

    q_columns = ['shot_id', 'distance', 'p', 'split', 'q_min_before', 'q_min_after', 'q_min_improved']
    e2_trials = pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_rows), E2_PATH)
    q_trials = pcm_fixed_points_save_csv(pd.DataFrame(q_rows, columns=q_columns), Q_PATH)

q_summary = q_trials.groupby(['distance', 'p'], as_index=False).agg(shots=('shot_id', 'size'), q_min_improvement_rate=('q_min_improved', 'mean')) if len(q_trials) else pd.DataFrame(columns=['distance', 'p', 'shots', 'q_min_improvement_rate'])
pcm_fixed_points_save_csv(q_summary, pcm_fixed_points_data_paths.output('E2_qmin_improvement_csv'))


In [ ]:
mechanism = b_analysis[['shot_id', 'margin_bin', 'pcm_sensitive', 'multi_basin', 'basin_state']].merge(pcm_fixed_points_gap_cases[['shot_id', 'message_instability']], on='shot_id')
pcm_fixed_points_development_instability = pcm_fixed_points_gap_cases[pcm_fixed_points_gap_cases.split == 'development'].message_instability
if pcm_fixed_points_config['smoke_mode'] and pcm_fixed_points_development_instability.empty:
    pcm_fixed_points_development_instability = pcm_fixed_points_gap_cases.message_instability
instability_threshold = float(pcm_fixed_points_development_instability.quantile(0.75))
mechanism['high_instability'] = (mechanism.message_instability >= instability_threshold).astype(int)
family_names = {
    'random_mixed': 'random_PCM',
    'ambiguity_targeted': 'ambiguity_PCM',
    'instability_targeted': 'instability_PCM',
    'same_PCM_restart': 'same_PCM_restart',
}

e3_pcm = e1_trials[e1_trials.family.isin(family_names)].copy()
e3_pcm['method'] = e3_pcm.family.map(family_names)
e3_osd = e2_trials[e2_trials.method.isin(['forced_endpoint_OSD1', 'forced_full_OSD'])].copy()
e3_osd['method'] = e3_osd.method.map({'forced_endpoint_OSD1': 'OSD1', 'forced_full_OSD': 'full_OSD'})
e3 = pd.concat([e3_pcm[['shot_id', 'method', 'logical_correct']], e3_osd[['shot_id', 'method', 'logical_correct']]], ignore_index=True)
e3 = e3[e3.shot_id.isin(mechanism.shot_id)].merge(mechanism, on='shot_id')
assert set(e3.method) <= {'random_PCM', 'ambiguity_PCM', 'instability_PCM', 'same_PCM_restart', 'OSD1', 'full_OSD'}
e3_rows = []

for grouping in ['margin_bin', 'pcm_sensitive', 'basin_state', 'high_instability']:
    aggregate = e3.groupby([grouping, 'method'], as_index=False).agg(cases=('shot_id', 'nunique'), repair_rate=('logical_correct', 'mean'))
    aggregate['mechanism_group'] = grouping
    aggregate = aggregate.rename(columns={grouping: 'group_value'})
    e3_rows.append(aggregate)

e3_summary = pcm_fixed_points_save_csv(pd.concat(e3_rows, ignore_index=True) if e3_rows else pd.DataFrame(columns=['group_value', 'method', 'cases', 'repair_rate', 'mechanism_group']), pcm_fixed_points_data_paths.output('E3_mechanism_conditional_repair_csv'))
final_methods = ['standard_BPOSD', 'Tesseract', 'best_PCM_only_BP', 'best_targeted_PCM_BP', 'best_PCM_plus_OSD1', 'forced_full_OSD']
pcm_fixed_points_final_trials = e2_trials[(e2_trials.split == 'test') & e2_trials.shot_id.isin(e4_population_ids) & e2_trials.method.isin(final_methods)].copy()
pcm_fixed_points_final_summary = pcm_fixed_points_final_trials.groupby(['distance', 'p', 'method'], as_index=False).agg(
    shots=('shot_id', 'nunique'),
    LER=('logical_correct', lambda x: 1 - x.mean()),
    syndrome_consistent_rate=('syndrome_consistent', 'mean'),
    mean_bp_runs=('bp_runs', 'mean'),
    mean_pcm_views=('pcm_views', 'mean'),
    mean_iterations=('iterations', 'mean'),
    fraction_OSD1=('fraction_OSD1', 'mean'),
    fraction_full_OSD=('fraction_full_OSD', 'mean'),
    runtime_samples=('runtime', 'count'),
    mean_runtime=('runtime', 'mean'),
    p95_runtime=('runtime', lambda x: x.quantile(0.95)),
)
metric_rows = []

for pcm_fixed_points_row in pcm_fixed_points_final_summary.itertuples(index=False):
    pcm_fixed_points_point = pcm_fixed_points_final_trials[(pcm_fixed_points_final_trials.distance == pcm_fixed_points_row.distance) & (pcm_fixed_points_final_trials.p == pcm_fixed_points_row.p)]
    baseline_ler = 1 - pcm_fixed_points_point[pcm_fixed_points_point.method == 'standard_BPOSD'].logical_correct.mean()
    tess_ler = 1 - pcm_fixed_points_point[pcm_fixed_points_point.method == 'Tesseract'].logical_correct.mean()
    pcm_fixed_points_denominator = baseline_ler - tess_ler
    pcm_fixed_points_method = pcm_fixed_points_point[pcm_fixed_points_point.method == pcm_fixed_points_row.method].set_index('shot_id')
    pcm_fixed_points_baseline = pcm_fixed_points_point[pcm_fixed_points_point.method == 'standard_BPOSD'].set_index('shot_id')
    gap = (pcm_fixed_points_baseline.logical_correct == 0) & (pcm_fixed_points_baseline.tesseract_correct == 1)
    correct_baseline = pcm_fixed_points_baseline.logical_correct == 1
    metric_rows.append({
        'distance': pcm_fixed_points_row.distance,
        'p': pcm_fixed_points_row.p,
        'method': pcm_fixed_points_row.method,
        'gap_closure': np.nan if np.isclose(pcm_fixed_points_denominator, 0) else (baseline_ler - pcm_fixed_points_row.LER) / pcm_fixed_points_denominator,
        'conditional_gap_repair': np.nan if not gap.any() else pcm_fixed_points_method.loc[gap].logical_correct.mean(),
        'regression': np.nan if not correct_baseline.any() else 1 - pcm_fixed_points_method.loc[correct_baseline].logical_correct.mean(),
    })

pcm_fixed_points_final_summary = pcm_fixed_points_final_summary.merge(pd.DataFrame(metric_rows), on=['distance', 'p', 'method'])
pcm_fixed_points_save_csv(pcm_fixed_points_final_summary, pcm_fixed_points_data_paths.output('E4_final_benchmark_csv'))
pcm_fixed_points_bootstrap_rows = []

for (pcm_fixed_points_distance, pcm_fixed_points_p), pcm_fixed_points_point in pcm_fixed_points_final_trials.groupby(['distance', 'p']):
    pcm_fixed_points_wide = pcm_fixed_points_point.pivot(index='shot_id', columns='method', values='logical_correct').sort_index()
    assert not pcm_fixed_points_wide.isna().any().any()
    pcm_fixed_points_rng = np.random.default_rng(pcm_fixed_points_config['random_seed'] + int(pcm_fixed_points_distance * 1000 + pcm_fixed_points_p * 1000000.0))
    pcm_fixed_points_indices = pcm_fixed_points_rng.integers(0, len(pcm_fixed_points_wide), size=(pcm_fixed_points_config['bootstrap_repetitions'], len(pcm_fixed_points_wide)))
    pcm_fixed_points_baseline_error = 1 - pcm_fixed_points_wide['standard_BPOSD'].to_numpy(float)

    for pcm_fixed_points_method in pcm_fixed_points_wide:
        pcm_fixed_points_errors = 1 - pcm_fixed_points_wide[pcm_fixed_points_method].to_numpy(float)
        samples = pcm_fixed_points_errors[pcm_fixed_points_indices].mean(axis=1)
        delta = samples - pcm_fixed_points_baseline_error[pcm_fixed_points_indices].mean(axis=1)
        pcm_fixed_points_bootstrap_rows.append({
            'distance': pcm_fixed_points_distance,
            'p': pcm_fixed_points_p,
            'method': pcm_fixed_points_method,
            'LER_ci_low': np.quantile(samples, 0.025),
            'LER_ci_high': np.quantile(samples, 0.975),
            'paired_delta_LER_ci_low': np.quantile(delta, 0.025),
            'paired_delta_LER_ci_high': np.quantile(delta, 0.975),
        })

pcm_fixed_points_save_csv(pd.DataFrame(pcm_fixed_points_bootstrap_rows), pcm_fixed_points_data_paths.output('E4_paired_bootstrap_csv'))
pcm_fixed_points_final_summary


## Figures 1–8

Plot the mechanism, ensemble and selection measurements computed above. Read preliminary diagnostic plots separately from the availability of final Part-E benchmark results.


In [ ]:
pcm_fixed_points_figure, pcm_fixed_points_axes = plt.subplots(2, 4, figsize=(18, 8))
sns.scatterplot(
    data=b_analysis,
    x='normalized_logical_margin',
    y='correct_fraction_redundant_addition',
    ax=pcm_fixed_points_axes[0, 0],
)
pcm_fixed_points_axes[0, 0].set_title('1 Margin vs PCM sensitivity')
sns.scatterplot(
    data=b_analysis,
    x='normalized_logical_margin',
    y='basin_entropy',
    ax=pcm_fixed_points_axes[0, 1],
)
pcm_fixed_points_axes[0, 1].set_title('2 Margin vs basin entropy')
sns.lineplot(
    data=a1_summary,
    x='p',
    y='oracle_generation_rate',
    hue='view_kind',
    marker='o',
    ax=pcm_fixed_points_axes[0, 2],
)
pcm_fixed_points_axes[0, 2].set_title('3 Basis vs redundant')

c_plot = pd.read_csv(pcm_fixed_points_data_paths.output('C1_C2_distinct_targeted_scaling_csv'))
sns.lineplot(
    data=c_plot[c_plot.construction == 'top_m_distinct'],
    x='view_budget',
    y='oracle_generation_rate',
    hue='strategy',
    ax=pcm_fixed_points_axes[0, 3],
)
pcm_fixed_points_axes[0, 3].set_title('4 Distinct targeted scaling')

d_plot = pd.read_csv(pcm_fixed_points_data_paths.output('D5_generation_selection_gap_csv'))
sns.lineplot(
    data=d_plot[d_plot.selector == BEST_SELECTOR],
    x='view_budget',
    y='oracle_generation_rate',
    hue='family',
    ax=pcm_fixed_points_axes[1, 0],
)
pcm_fixed_points_axes[1, 0].set_title('5 Generation vs selection')
sns.lineplot(
    data=d_plot[d_plot.selector == BEST_SELECTOR],
    x='view_budget',
    y='selected_success_rate',
    hue='family',
    legend=False,
    linestyle='--',
    ax=pcm_fixed_points_axes[1, 0],
)

d4_plot = pd.read_csv(pcm_fixed_points_data_paths.output('D4_selector_margin_csv'))
sns.barplot(
    data=d4_plot,
    x='margin_bin',
    y='selector_accuracy',
    hue='selector',
    ax=pcm_fixed_points_axes[1, 1],
)
pcm_fixed_points_axes[1, 1].tick_params(axis='x', rotation=45)
pcm_fixed_points_axes[1, 1].set_title('6 Selector vs margin')

if len(b3):
    pcm_fixed_points_axes[1, 2].scatter(b3.standard, b3.successful)
    pcm_fixed_points_axes[1, 2].plot([0, 1], [0, 1], color='black')
pcm_fixed_points_axes[1, 2].set_title('7 Basin P(true): standard vs successful')
sns.lineplot(
    data=pcm_fixed_points_final_summary,
    x='p',
    y='LER',
    hue='method',
    style='distance',
    marker='o',
    ax=pcm_fixed_points_axes[1, 3],
)
pcm_fixed_points_axes[1, 3].set_title('8 Final LER')
plt.tight_layout()

plt.show()


## Completeness and semantic assertions


In [ ]:
required_stats = ['A1_basis_vs_redundant.csv', 'A2_paired_global.csv', 'A2_paired_local.csv', 'B1_B2_margin_correlations.csv', 'B1_B2_margin_bins.csv', 'B1_cohort_margins.csv', 'B3_basin_probability_shift.csv', 'B4_margin_oscillation_correlation.csv', 'B4_grouped_oscillation.csv', 'C1_C2_distinct_targeted_scaling.csv', 'C2_stratified_generation.csv', 'C3_row_coverage_repair.csv', 'D1_D2_generation_diversity.csv', 'D2_margin_sector_entropy.csv', 'D3_selector_tuning.csv', 'D3_frozen_selector.csv', 'D4_selector_margin.csv', 'D5_generation_selection_gap.csv', 'E1_pcm_ensemble_summary.csv', 'E2_qmin_improvement.csv', 'E3_mechanism_conditional_repair.csv', 'E4_final_benchmark.csv', 'E4_paired_bootstrap.csv']
assert all((pcm_fixed_points_data_paths.output('name', name=name).is_file() for name in required_stats))
assert pcm_fixed_points_policy.selection_split.eq('development').all()
assert set(pcm_fixed_points_final_trials.method) == set(final_methods)
assert pcm_fixed_points_final_trials.groupby(['distance', 'p', 'method']).shot_id.nunique().groupby(level=[0, 1]).nunique().eq(1).all()
pcm_fixed_points_expected_distances = {pcm_fixed_points_context_for(level, pcm_fixed_points_config['p_values'][0]).distance for level in pcm_fixed_points_config['primary_levels'] + pcm_fixed_points_config['validation_levels']}
assert set(pcm_fixed_points_final_trials.distance) == pcm_fixed_points_expected_distances
assert pcm_fixed_points_final_trials.shot_id.isin(unbiased[unbiased.split == 'test'].shot_id).all()
statuses = []
a2_eligible = a1_views.groupby(['shot_id', 'view_kind']).logical_correct.agg(['min', 'max'])
a2_eligible = int(((a2_eligible['min'] == 0) & (a2_eligible['max'] == 1)).sum())
basin_eligible_cases = int(pcm_fixed_points_restart_trials.loc[pcm_fixed_points_restart_trials.basin_eligible == 1, 'shot_id'].nunique())
e_population_count = int(e1_trials.loc[e1_trials.family == 'standard_BP', 'shot_id'].nunique())
final_population_count = int(pcm_fixed_points_final_trials.shot_id.nunique())
frames = [('A1', a1_views, len(pcm_fixed_points_gap_cases)), ('A2', a2_pairs, a2_eligible), ('B1', b_analysis, len(pcm_fixed_points_gap_cases)), ('B2', basin_summary, basin_eligible_cases), ('B3', b3, len(b3)), ('B4', b4, len(pcm_fixed_points_gap_cases)), ('C1', targeted_candidates, len(pcm_fixed_points_gap_cases)), ('C2', c_prefix, len(pcm_fixed_points_gap_cases)), ('C3', row_trials, len(pcm_fixed_points_gap_cases)), ('D1', d_prefix, len(pcm_fixed_points_gap_cases)), ('D2', d2_margin, len(pcm_fixed_points_gap_cases)), ('D3', selector_trials, len(pcm_fixed_points_gap_cases)), ('D4', d4, len(pcm_fixed_points_gap_cases)), ('D5', selector_trials, len(pcm_fixed_points_gap_cases)), ('E1', e1_trials, e_population_count), ('E2', e2_trials, e_population_count), ('E3', e3, e3.shot_id.nunique()), ('E4', pcm_fixed_points_final_trials, final_population_count)]

for pcm_fixed_points_name, pcm_fixed_points_frame, pcm_fixed_points_eligible in frames:
    status = 'no_eligible_cases' if pcm_fixed_points_eligible == 0 else 'completed' if len(pcm_fixed_points_frame) else 'missing'
    statuses.append({
        'experiment': pcm_fixed_points_name,
        'implemented': int(status != 'missing'),
        'execution_status': status,
        'eligible_cases': int(pcm_fixed_points_eligible),
        'trace_rows': len(pcm_fixed_points_frame),
    })

pcm_fixed_points_experiment_status = pcm_fixed_points_save_csv(pd.DataFrame(statuses), pcm_fixed_points_data_paths.output('experiment_status_csv'))
assert pcm_fixed_points_experiment_status.implemented.eq(1).all(), pcm_fixed_points_experiment_status
pcm_fixed_points_log(2, {
    'status': 'smoke_passed' if pcm_fixed_points_config['smoke_mode'] else 'complete',
    'experiments': pcm_fixed_points_experiment_status.experiment.tolist(),
    'gap_cases': len(pcm_fixed_points_gap_cases),
    'best_selector': BEST_SELECTOR,
    'best_family': best_family,
    'best_targeted_family': targeted_policy,
})
pcm_fixed_points_experiment_status
